#### Application Codebase Knowledge Graph (KG) Tools

##### Environment setup

###### Package imports

In [2]:
import os
import sys
from dotenv import load_dotenv
import time
import requests
import pandas as pd
from pathlib import Path
import importlib
from neo4j import GraphDatabase
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget
import gradio as gr
import gradio.blocks

# Load environment variables
env_file=Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE")

###### Test connections

In [9]:
def test_neo4j_connection(env_file=Path.cwd().parent / ".env"):
    """
    Test connectivity to a Neo4j database using settings
    stored in a .env file.

    Expected variables:

        NEO4J_URI
        NEO4J_USERNAME
        NEO4J_PASSWORD
        NEO4J_DATABASE
    """

    try:
        with GraphDatabase.driver(uri, auth=(username, password), database=database) as driver:

            # Test the connection
            driver.verify_connectivity()

            # Open a session
            with driver.session(database=database) as session:
                result = session.run(
                    """
                    RETURN
                        1 AS connected,
                        datetime() AS server_time
                    """
                )

                record = result.single()

                print("✅ Successfully connected to Neo4j")
                print(f"Database    : {database}")
                print(f"URI         : {uri}")
                print(f"Server Time : {record['server_time']}")

                return True

    except Exception as ex:
        print("❌ Connection failed")
        print(type(ex).__name__)
        print(ex)

        return False

test_neo4j_connection()

✅ Successfully connected to Neo4j
Database    : appkg
URI         : bolt://localhost:7687
Server Time : 2026-11-11T04:27:32.264000000+00:00


True

###### Create Neo4j driver object and python-cypher method

In [10]:
# Create driver object
# Keeping the Neo4j Driver open for the lifetime of your Jupyter session is the normal pattern and does not, by itself, constitute a memory leak.
driver = GraphDatabase.driver(uri, auth=(username, password), database=database)

driver.verify_connectivity()

# Define a generic function to run a Cypher query
def run_cypher_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return result.data()

###### Code: Check pre-existing Neo4j graph schema

In [20]:
# Generate graph from Cypher query
g = Neo4jGraphWidget(driver)
# View schema
g.show_cypher("CALL db.schema.visualization()")

query_node_frequency = """
MATCH (n)
RETURN labels(n) AS label, count(*) AS count
ORDER BY count DESC;
"""
result_query_node_frequency = run_cypher_query(query_node_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_nodes = pd.DataFrame(result_query_node_frequency)
# Flatten the 'label' column since it's a list
df_nodes['label'] = df_nodes['label'].apply(
    lambda x: x[0] if len(x) > 0 else None)
print("Nodes by Count")
print(df_nodes)  # Outputs the list of nodes (Persons)

query_relationship_frequency = """
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, COUNT(r) AS count
ORDER BY count DESC;
"""
result_query_relationship_frequency = run_cypher_query(
    query_relationship_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_rels = pd.DataFrame(result_query_relationship_frequency)
# Flatten the 'label' column since it's a list
# df_rels['label'] = df_rels['label'].apply(lambda x: x[0] if len(x) > 0 else None)
print("Relationships by Count")
print(df_rels)  # Outputs the list of nodes (Persons)

GraphWidget(layout=Layout(height='500px', width='100%'))

Nodes by Count
       label  count
0   FileName   1634
1  Directory    271
2   FileType     47
Relationships by Count
  relationshipType  count
0         CONTAINS   3489
1       IS_TYPE_OF   1634


###### No. of FileNames by FileType

In [46]:
query_count_of_filetypes = """
MATCH (f:FileName)-[:IS_TYPE_OF]->(t:FileType)
RETURN t.type AS FileType, COUNT(f) AS FileCount
ORDER BY FileCount DESC;
"""

result_query_count_of_filetypes = run_cypher_query(query_count_of_filetypes)
df_result_query_count_of_filetypes = pd.DataFrame(result_query_count_of_filetypes)
print(f"Count of FileTypes:\n{df_result_query_count_of_filetypes}\n")
total_files = df_result_query_count_of_filetypes["FileCount"].sum()
print(f"\nTotal number of files: {total_files}")

Count of FileTypes:
                    FileType  FileCount
0                 Javascript        555
1                        SVG        319
2                        CSS        160
3                     Header        103
4                      Other         97
5                        C++         77
6                      XHTML         51
7                        PNG         37
8                       HTML         23
9                       JSON         23
10                       PDF         18
11                      Text         16
12  FreeMarker_Java_template         15
13                Typescript         14
14               DTD_XML_def         12
15                     Shell         12
16                       IDL         10
17                       RSS          7
18                      Epub          7
19                       NSH          7
20                       GIF          7
21                       XUL          6
22                 Java_prop          5
23                  

##### Mapping of FileType 'Other' by Extension

In [47]:
query_count_of_filetype_ext = """
MATCH (f:FileName)-[:IS_TYPE_OF]->(t:FileType)
WHERE t.type = 'Other'
WITH f,
     CASE
       WHEN f.name CONTAINS '.' THEN toLower(last(split(f.name, '.')))
       ELSE 'No Extension'
     END AS Extension
RETURN Extension, COUNT(f) AS FileCount
ORDER BY FileCount DESC;
"""

result_query_count_of_filetype_ext = run_cypher_query(query_count_of_filetype_ext)
df_result_query_count_of_filetype_ext = pd.DataFrame(result_query_count_of_filetype_ext)
print(df_result_query_count_of_filetype_ext)

total_files = df_result_query_count_of_filetype_ext["FileCount"].sum()
print(f"\nTotal number of files: {total_files}")

                     Extension  FileCount
0                 No Extension         63
1                          mts          6
2                           xz          5
3                        patch          3
4                          rtf          2
5                          ris          1
6                         ctv6          1
7                      babelrc          1
8                gitattributes          1
9                    gitignore          1
10                  gitmodules          1
11                     desktop          1
12                       plist          1
13                         car          1
14  0_release_build_and_deploy          1
15                         sfx          1
16                         def          1
17                         dsp          1
18                         dsw          1
19                         tag          1
20                         xsl          1
21                    eslintrc          1
22                         ttf    

###### Test Github API connection

In [4]:
def test_github_connection():
    """Test connectivity to GitHub and validate the GitHub PAT."""

    # Find .env in the directory above the notebook's working directory
    env_file = Path.cwd().parent / ".env"

    print(f"Loading environment from: {env_file}")

    if not env_file.exists():
        print("❌ .env file not found")
        return False

    load_dotenv(env_file, override=True)

    token = os.getenv("GITHUB_TOKEN")

    if not token:
        print("❌ GITHUB_TOKEN is missing from .env")
        return False

    # GitHub API headers
    headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {token}",
        "X-GitHub-Api-Version": "2026-03-10",
    }

    try:
        # Test GitHub API + PAT authentication
        response = requests.get(
            "https://api.github.com/user",
            headers=headers,
            timeout=10,
        )

        print(f"GitHub API status: {response.status_code}")

        if response.status_code == 200:
            user = response.json()

            print("✅ Successfully connected to GitHub")
            print("✅ PAT is valid")
            print(f"GitHub user : {user.get('login')}")
            print(f"User ID     : {user.get('id')}")
            print(f"Account type: {user.get('type')}")

            return True

        elif response.status_code == 401:
            print("❌ GitHub API connection succeeded")
            print("❌ PAT authentication failed")
            print("The token may be invalid, expired, or revoked.")

        elif response.status_code == 403:
            print("⚠️ GitHub API responded with 403 Forbidden")
            print("The PAT was received, but GitHub denied the request.")
            print(response.text)

        else:
            print("❌ GitHub API request failed")
            print(response.text)

        return False

    except requests.exceptions.Timeout:
        print("❌ Connection to GitHub timed out")
        return False

    except requests.exceptions.ConnectionError as ex:
        print("❌ Could not connect to GitHub")
        print(ex)
        return False

    except requests.exceptions.RequestException as ex:
        print("❌ GitHub request failed")
        print(type(ex).__name__)
        print(ex)
        return False

test_github_connection()

Loading environment from: E:\Projects\knowledge_graph_tools\.env
GitHub API status: 200
✅ Successfully connected to GitHub
✅ PAT is valid
GitHub user : georgejaymcmc
User ID     : 88515517
Account type: User


True

##### Repo details

In [5]:
# 1. Enable auto-reloading so changes to your .py file update automatically
%load_ext autoreload
%autoreload 2

# 2. Add your project 'src' directory to Python's path
# Adjust the number of parents depending on where your notebook is located
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parents[0] # Assumes notebook is one folder deep (e.g., in a 'notebooks/' folder)
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 3. Import your service directly from your local module
from codebase_kg.services.github_repo_service import GithubRepoService

# 4. Initialize and use it
# service = GithubRepoService(token="YOUR_GITHUB_TOKEN")
print("Module successfully imported from:", sys.modules['codebase_kg.services.github_repo_service'].__file__)


Module successfully imported from: E:\Projects\knowledge_graph_tools\src\codebase_kg\services\github_repo_service.py


##### Extracting Zotero repo stats

###### Code: Choose repo to analyse
1. Fork repo to baseline codebase - 16/09/2026

In [147]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "georgejaymcmc"
repo = "zotero"

# Print result
result = github.stats(owner, repo)

# Pretty print result
for key, value in result.items():
    print(f"{key:20}: {value}")


name                : zotero
full_name           : georgejaymcmc/zotero
description         : Zotero is a free, easy-to-use tool to help you collect, organize, annotate, cite, and share your research sources.
language            : None
size_kb             : 242678
visibility          : public
default_branch      : main
stars               : 0
forks               : 0
watchers            : 0
directories         : 363
files               : 3489
url                 : https://github.com/georgejaymcmc/zotero


###### Code: Purge the cached module from the notebook

In [148]:
# 1. Collect all loaded submodules related to your project package
to_delete = [name for name in sys.modules if name.startswith("codebase_kg")]

# 2. Purge them from the sys.modules memory cache
for module_name in to_delete:
    del sys.modules[module_name]
    print(f"🔥 Purged cache for: {module_name}")

# 3. Re-import your updated class freshly from the disk
from codebase_kg.services.github_repo_service import GithubRepoService


🔥 Purged cache for: codebase_kg
🔥 Purged cache for: codebase_kg.services
🔥 Purged cache for: codebase_kg.services.github_repo_service


###### Code: Output repo file types and directory location to csv

In [6]:
# Load Github PAT
env_file = Path.cwd().parent / ".env"
load_dotenv(env_file, override=True)

github_token = os.getenv("GITHUB_TOKEN")

# Create the service
github = GithubRepoService(token=github_token)

# Repository name
owner = "georgejaymcmc"
repo = "zotero"


In [150]:
# 1. RUN THIS METHOD TO GENERATE THE CSV FILE
print("Generating CSV file...")
csv_results = github.file_types(owner, repo, output_csv="zotero_files.csv")

# 2. Pretty print the execution summary returned by file_types
print("\n--- Execution Summary ---")
for key, value in csv_results.items():
    print(f"{key:20}: {value}")


Generating CSV file...
📦 Dataframe successfully dumped to: E:\Projects\knowledge_graph_tools\src\codebase_kg\neo4j_imports\zotero_files.csv

--- Execution Summary ---
total_files         : 3489
directory_count     : 271
output_csv          : zotero_files.csv
file_type_counts    : {'SVG': 774, 'FreeMarker_Java_template': 673, 'Javascript': 578, 'DTD_XML_def': 482, 'CSS': 202, 'Java_prop': 146, 'Other': 108, 'Header': 107, 'C++': 77, 'JSON': 63, 'XHTML': 51, 'PNG': 41, 'HTML': 23, 'PDF': 18, 'Text': 16, 'Typescript': 14, 'Shell': 12, 'IDL': 10, 'NSH': 7, 'GIF': 7, 'RSS': 7, 'Epub': 7, 'Markdown': 6, 'RC': 6, 'XUL': 6, 'NSI': 5, 'XML': 4, 'Python Script': 4, 'ICO': 4, 'SQL': 4, '.ini': 3, '.manifest': 3, 'CSL': 3, 'VBScript': 2, 'WOFF': 2, 'SQLite': 2, 'RDF': 2, 'YAML': 1, 'C': 1, 'NLF': 1, 'XPI': 1, 'ATOM': 1, 'OPML': 1, 'LUA': 1, 'OPF': 1, 'JPG': 1, 'ZIP': 1}
unknown_extensions  : ['', '.0_release_build_and_deploy', '.car', '.ctv6', '.def', '.desktop', '.dsp', '.dsw', '.mts', '.patch', 

##### Import what we know into Neo4j: KGs are built progressively
- directories, file names and file types form the base of the App KG

###### Start from scratch: Delete all nodes, relationships, etc from Neo4j

In [142]:
# Delete all nodes and relationships

query_delete_all_nodes_relationships = """
MATCH (n)
DETACH DELETE n
RETURN count(n) AS deleted_nodes
"""

result_query_delete_all_nodes_relationships = run_cypher_query(query_delete_all_nodes_relationships)

result_query_delete_all_nodes_relationships

[{'deleted_nodes': 0}]

###### Upload repo_files.csv into Neo4j to create base graph elements

In [151]:
# Generic function to run Cypher queries
query_load_repo_files = """
// Load File Type, Directory, and File Name nodes
LOAD CSV WITH HEADERS FROM 'file:///zotero_files.csv' AS row
MERGE (f:FileName {name: row.FileName})
MERGE (d:Directory {name: row.Directory})
MERGE (t:FileType {type: row.FileType})
MERGE (d)-[:CONTAINS]->(f)
MERGE (f)-[:IS_TYPE_OF]->(t);
"""
result_query_load_repo_files = run_cypher_query(query_load_repo_files)

###### Check the upload
- Compare file count with above repo query result

In [48]:
# Generate graph from Cypher query
g = Neo4jGraphWidget(driver)
# View schema
g.show_cypher("CALL db.schema.visualization()")

query_node_frequency = """
MATCH (n)
RETURN labels(n) AS label, count(*) AS count
ORDER BY count DESC;
"""
result_query_node_frequency = run_cypher_query(query_node_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_nodes = pd.DataFrame(result_query_node_frequency)
# Flatten the 'label' column since it's a list
df_nodes['label'] = df_nodes['label'].apply(
    lambda x: x[0] if len(x) > 0 else None)
print("Nodes by Count")
print(df_nodes)  # Outputs the list of nodes (Persons)

query_relationship_frequency = """
MATCH ()-[r]->()
RETURN type(r) AS relationshipType, COUNT(r) AS count
ORDER BY count DESC;
"""
result_query_relationship_frequency = run_cypher_query(
    query_relationship_frequency)
# Convert the list of dictionaries to a pandas DataFrame
df_rels = pd.DataFrame(result_query_relationship_frequency)
# Flatten the 'label' column since it's a list
# df_rels['label'] = df_rels['label'].apply(lambda x: x[0] if len(x) > 0 else None)
print("Relationships by Count")
print(df_rels)  # Outputs the list of nodes (Persons)


GraphWidget(layout=Layout(height='500px', width='100%'))

Nodes by Count
       label  count
0   FileName   1634
1  Directory    271
2   FileType     47
Relationships by Count
  relationshipType  count
0         CONTAINS   3489
1       IS_TYPE_OF   1634


##### Gradio chatbot file selector

###### Function to list labels and relationship types

In [49]:
def get_graph_schema():
    """Return node labels and relationship types with counts as a DataFrame."""

    try:

        # ---------------------------------
        # Node labels and properties
        # ---------------------------------

        labels = run_cypher_query("""
            MATCH (n)
            UNWIND labels(n) AS label
            UNWIND keys(n) AS property
            RETURN
                label,
                count(DISTINCT n) AS count,
                collect(DISTINCT property) AS properties
            ORDER BY label
        """)

        # ---------------------------------
        # Relationship types
        # ---------------------------------

        relationships = run_cypher_query("""
            MATCH ()-[r]->()
            RETURN
                type(r) AS relationship,
                count(*) AS count
            ORDER BY relationship
        """)

        # ---------------------------------
        # Build combined result
        # ---------------------------------

        rows = []

        # Nodes
        for row in labels:
            rows.append({
                "Type": "Node",
                "Name": row["label"],
                "Count": row["count"],
                "Properties": ", ".join(
                    sorted(row["properties"])
                )
            })

        # Relationships
        for row in relationships:
            rows.append({
                "Type": "Relationship",
                "Name": row["relationship"],
                "Count": row["count"],
                "Properties": ""
            })

        # ---------------------------------
        # Return DataFrame
        # ---------------------------------

        return pd.DataFrame(
            rows,
            columns=[
                "Type",
                "Name",
                "Count",
                "Properties"
            ]
        )

    except Exception as e:

        return pd.DataFrame({
            "Error": [
                f"{type(e).__name__}: {e}"
            ]
        })

###### Convert cypher query results to pandas df

In [50]:
def execute_cypher(query):
    """Execute Cypher and return the results as a DataFrame."""

    if not query or not query.strip():
        return pd.DataFrame({
            "Result": ["Please enter a Cypher query."]
        })

    try:
        records = run_cypher_query(query)

        if not records:
            return pd.DataFrame({
                "Result": ["Query returned no rows."]
            })

        return pd.DataFrame(records)

    except Exception as e:
        return pd.DataFrame({
            "Error": [
                f"{type(e).__name__}: {e}"
            ]
        })

###### Gradio interface

###### Test queries

In [51]:
query_test="""
MATCH (d:Directory)-[:CONTAINS]->(f:FileName)-[:IS_TYPE_OF]->(ft:FileType)
WHERE ft.type = "Javascript"

WITH f.name AS file_name, count(DISTINCT d) AS directory_count
WHERE directory_count > 1

MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
WHERE f.name = file_name

RETURN
    file_name,
    d.name AS directory
ORDER BY file_name, directory
"""

result_query_test = run_cypher_query(query_test)

result_query_test

[{'file_name': 'commandLineHandler.js', 'directory': 'app/assets'},
 {'file_name': 'commandLineHandler.js',
  'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'contextPane.js', 'directory': 'chrome/content/zotero'},
 {'file_name': 'contextPane.js',
  'directory': 'chrome/content/zotero/elements'},
 {'file_name': 'http.js',
  'directory': 'chrome/content/zotero/actors/translation'},
 {'file_name': 'http.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'id.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'id.js', 'directory': 'resource/tinymce/langs'},
 {'file_name': 'locale.js', 'directory': 'chrome/content/zotero/xpcom'},
 {'file_name': 'locale.js', 'directory': 'resource/tinymce'},
 {'file_name': 'plugin.min.js',
  'directory': 'resource/tinymce/plugins/autolink'},
 {'file_name': 'plugin.min.js', 'directory': 'resource/tinymce/plugins/code'},
 {'file_name': 'plugin.min.js',
  'directory': 'resource/tinymce/plugins/contextmenu'},
 {'file_name'

In [52]:

with gr.Blocks(title="Neo4j Cypher Explorer") as demo:
    gr.Markdown("# Neo4j Cypher Explorer")
    gr.Markdown(
        "Explore the current graph model and execute Cypher queries."
    )

    # ---------------------------------------------------------
    # Graph schema
    # ---------------------------------------------------------

    gr.Markdown("## Graph Schema")

    schema_button = gr.Button(
        "Refresh Graph Schema",
        variant="secondary"
    )

    schema_output = gr.Dataframe(
        label="Graph Schema",
        interactive=False,
        wrap=True
    )

    # ---------------------------------------------------------
    # Cypher query
    # ---------------------------------------------------------

    gr.Markdown("## Cypher Query")

    cypher_input = gr.Textbox(
        label="Enter Cypher",
        lines=8,
        value="""
        MATCH (d:Directory)-[:CONTAINS]->(f:FileName)-[:IS_TYPE_OF]->(ft:FileType)
WHERE ft.type = "Javascript"

WITH f.name AS file_name, count(DISTINCT d) AS directory_count
WHERE directory_count > 1

MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
WHERE f.name = file_name

RETURN
    file_name,
    d.name AS directory
ORDER BY file_name, directory
    """
    )

    execute_button = gr.Button(
        "Run Query",
        variant="primary"
    )

    # ---------------------------------------------------------
    # Results
    # ---------------------------------------------------------

    gr.Markdown("## Results")

    result_output = gr.Dataframe(
        label="Query Result",
        interactive=False,
        wrap=True
    )

    # ---------------------------------------------------------
    # Events
    # ---------------------------------------------------------

    schema_button.click(
        fn=get_graph_schema,
        outputs=schema_output
    )

    execute_button.click(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )

    cypher_input.submit(
        fn=execute_cypher,
        inputs=cypher_input,
        outputs=result_output
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


##### Display repo file contents
```mermaid
sequenceDiagram
    participant A as Python CLI
    participant B as NEO4J
    participant C as GITHUB

    A->>B: 1. Query Neo4j for list of files
    B->>A: 2. List files in a Gradio SelectBox
    A->>C: 3. Select file to get syntax
    C->>A: 4. Return file contents for display in Gradio
```

###### 1. Query Neo4j for files

In [53]:
def get_files_from_neo4j(file_type=None):
    """Return files from Neo4j, optionally filtered by file type."""

    query = """
    MATCH (d:Directory)-[:CONTAINS]->(f:FileName)
          -[:IS_TYPE_OF]->(ft:FileType)

    WHERE $file_type IS NULL
       OR ft.type = $file_type

    RETURN
        d.name AS directory,
        f.name AS file_name,
        ft.type AS file_type

    ORDER BY directory, file_name
    """

    records = run_cypher_query(
        query,
        {"file_type": file_type}
    )

    return pd.DataFrame(records)

###### 2. Create Gradio file choices

In [54]:
def build_github_path(directory, file_name):
    """Convert Neo4j directory/file information into a GitHub path."""

    directory = str(directory).strip()

    if directory.upper() == "ROOT":
        return file_name

    return f"{directory}/{file_name}"


def get_file_choices(file_type=None):
    """Build Gradio file choices from Neo4j."""

    df = get_files_from_neo4j(file_type)

    if df.empty:
        return []

    choices = []

    for _, row in df.iterrows():

        file_path = build_github_path(
            row["directory"],
            row["file_name"]
        )

        choices.append((file_path, file_path))

    return choices

In [55]:
def get_file_types():
    """Return file types available in the Neo4j codebase."""

    query = """
    MATCH (ft:FileType)
    RETURN DISTINCT ft.type AS file_type
    ORDER BY file_type
    """

    records = run_cypher_query(query)

    return [
        row["file_type"]
        for row in records
        if row["file_type"]
    ]

In [56]:
def update_file_dropdown(file_type):
    """Update the file dropdown based on the selected file type."""

    if not file_type:
        return gr.Dropdown(
            choices=[],
            value=None
        )

    choices = get_file_choices(file_type)

    return gr.Dropdown(
        choices=choices,
        value=None
    )

###### 3. Map Neo4j FileType to `gr.Code` languages

In [57]:
from pathlib import Path


def get_language(file_name):
    """Return the Gradio language for a filename."""

    extension = Path(file_name).suffix.lower()

    language_map = {
        ".js": "javascript",
        ".jsx": "javascript",
        ".ts": "typescript",
        ".tsx": "typescript",

        ".py": "python",

        ".json": "json",

        ".html": "html",
        ".htm": "html",

        ".css": "css",

        ".sql": "sql",

        ".java": "java",

        ".c": "c",
        ".h": "c",
        ".cpp": "cpp",
        ".hpp": "cpp",

        ".cs": "csharp",

        ".md": "markdown",

        ".xml": "xml",

        ".yaml": "yaml",
        ".yml": "yaml",

        ".sh": "shell",

        ".txt": "text",
    }

    return language_map.get(extension, "text")

###### 4. Retrieve code from Github

In [65]:

def get_selected_file(file_path):
    """Retrieve the selected file from GitHub."""
    if not file_path:
        return gr.Code(value="", language=None)

    df = get_files_from_neo4j()

    matches = df[
        df.apply(
            lambda row: build_github_path(
                row["directory"],
                row["file_name"]
            ) == file_path,
            axis=1
        )
    ]

    if matches.empty:
        return gr.Code(value="", language=None)

    file_name = matches.iloc[0]["file_name"]
    language = get_language(file_name)

    # Gradio uses None for plain text instead of "text"
    if language == "text":
        language = None

    code = github.get_file_content(
        owner,
        repo,
        file_path
    )

    return gr.Code(value=str(code), language=language)

###### 5. Gradio UI

In [ ]:
with gr.Blocks(title="Code Explorer") as code_demo:

    gr.Markdown("# Code Explorer")
    gr.Markdown(
        "Select a file type, then select a file from the Neo4j codebase graph."
    )

    file_type_dropdown = gr.Dropdown(
        label="Select File Type",
        choices=get_file_types(),
        value=None,
        interactive=True
    )

    file_dropdown = gr.Dropdown(
        label="Select File",
        choices=[],
        value=None,
        interactive=True
    )

    code_output = gr.Code(
        label="Source Code",
        language="javascript",
        interactive=False,
        lines=40,
        max_lines=60
    )

    file_type_dropdown.change(
        fn=update_file_dropdown,
        inputs=file_type_dropdown,
        outputs=file_dropdown
    )

    file_dropdown.change(
        fn=get_selected_file,
        inputs=file_dropdown,
        outputs=code_output
    )

code_demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


###### Testing/debugging key Gradio functions

`def get_file_content(self, owner, repo, file_path)`

In [ ]:
github.get_file_content('georgejaymcmc', 'zotero', 'chrome/content/zotero-platform/unix/integration.css' )

In [185]:
get_selected_file('chrome/content/zotero-platform/unix/integration.css')

('window.citation-dialog {\n\t-moz-appearance: none;\n\tbackground: -moz-linear-gradient(-90deg, rgb(243,123,119) 0, rgb(180,47,38) 50%, rgb(156,36,27) 50%) !important;\n}\n\nwindow.citation-dialog.note-dialog {\n\tbackground: -moz-linear-gradient(-90deg, rgb(249, 231, 179) 0, rgb(228, 193, 94) 50%, rgb(221, 184, 81) 50%) !important;\n}\n\n.citation-dialog.entry {\n\tpadding: 10px;\n}',
 'css')

In [104]:
import inspect

print(inspect.getsource(github._repo_data))
print(type(github._repo_data(owner, repo)))

    def _repo_data(self, owner, repo):
        """Get repository metadata, using the cache when available."""

        cache_key = f"{owner}/{repo}"

        if cache_key not in self._repo_cache:
            url = f"https://api.github.com/repos/{owner}/{repo}"

            response = requests.get(
                url,
                headers=self.headers,
                timeout=10
            )

            if response.status_code != 200:
                raise RuntimeError(
                    f"GitHub API error {response.status_code}: "
                    f"{response.text}"
                )

            self._repo_cache[cache_key] = response.json()

        return self._repo_cache[cache_key]

<class 'dict'>


##### Application code syntax analysis overview
```mermaid
flowchart LR
subgraph App KG
    B(Application code)
    B-->|Extract|L(&#x2460 AST)
    B-->|Extract|M(&#x2461 Lexical search)
    O(Application KG Model)
    L-->O
    M-->O
    O-->B
end
```

Two types of analysis are required
1. Abstract Syntax Trees (AST)
   - using language-specific parsers to determine classes, functions, symbols, within each file
   - control flow graphs
   - call graphs but only for files from the same language family
2. Lexical search
   - to see which files and/or functions are called between languages
      - e.g. `javascript` files in `html`

##### Javascript ASTs - Using the `tree-sitter` language parse library

**Workflow**
1. Environment and configuration
   - Install dependencies, configure paths and initialise the parser.
2. Repository discovery
   - Identify JavaScript files and build files_df.
3. AST parsing
   - Parse each file and retain its syntax tree and parsing diagnostics.
4. AST utilities
   - Create reusable functions for traversing nodes, extracting source text and tracking locations.
5. Symbol extraction
   - Extract declarations into symbols_df.
6. Scope extraction
   - Build lexical scope information into scopes_df.
7. Import and export extraction
   - Build imports_df and exports_df.
8. Reference and assignment extraction
   - Build references_df and assignments_df.
9. Function call extraction
   - Build calls_df, preserving raw call expressions and caller context.
10. Class and inheritance extraction
    - Build inheritance_df and extend symbol metadata for classes and members.
11. Symbol resolution
    - Resolve references, imports, member access and calls into additional DataFrames.
12. Validation and reporting
    - Review extraction coverage, unresolved references and parsing errors.
13. Neo4j graph construction
    - Persist the code model and relationships with provenance.

###### 1. Environment and configuration

In [7]:
from __future__ import annotations

import hashlib
import json
import os
import re
from pathlib import Path
from collections import defaultdict
from typing import Any, Iterable

import pandas as pd
from tree_sitter_language_pack import get_parser

# Create the GitHub service
github = GithubRepoService(token=github_token)

# Repository details
owner = "georgejaymcmc"
repo = "zotero"

# Repository identifier
REPOSITORY_ID = f"{owner}/{repo}"

# Repository revision (optional)
REPOSITORY_REVISION = None

# Start with these extensions. Add TypeScript later if required.
SOURCE_EXTENSIONS = {".js", ".jsx", ".mjs", ".cjs"}

EXCLUDED_DIRS = {
    ".git", "node_modules", "dist", "build", "coverage",
    ".venv", "__pycache__", "vendor"
}

REPOSITORY_REVISION = None  # Set to a Git commit hash when available.

parser = get_parser("javascript")


def stable_id(*parts: Any, prefix: str = "") -> str:
    """Create a deterministic identifier from stable input values."""
    raw = "|".join("" if p is None else str(p) for p in parts)
    digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]
    return f"{prefix}{digest}"


def node_text(node, source_code: bytes) -> str:
    """Return the exact source text covered by a Tree-sitter node."""
    return source_code[node.start_byte:node.end_byte].decode("utf-8", errors="replace")


def node_location(node) -> dict:
    return {
        "start_line": node.start_point.row + 1,
        "start_column": node.start_point.column + 1,
        "end_line": node.end_point.row + 1,
        "end_column": node.end_point.column + 1,
        "start_byte": node.start_byte,
        "end_byte": node.end_byte,
    }


def walk_tree(node):
    """Yield a node and all descendants in source-tree order."""
    yield node
    for child in node.children:
        yield from walk_tree(child)


###### 2. Repository discovery

In [8]:
def discover_repository() -> pd.DataFrame:
    """
    Discover supported source files in the GitHub repository.
    """

    repository_files = github.get_repository_files(owner, repo)

    records = []

    for file_info in repository_files:

        rel = file_info["path"]
        relative = Path(rel)
        extension = file_info["extension"]

        # Only include supported JavaScript files
        if extension not in SOURCE_EXTENSIONS:
            continue

        # Exclude generated, dependency and build directories
        if any(part in EXCLUDED_DIRS for part in relative.parts):
            continue

        # Exclude resource placeholder files from AST extraction
        if relative.parts[0] == "resource":
            continue

        records.append({
            "file_id": stable_id(
                REPOSITORY_ID,
                rel,
                prefix="file_"
            ),
            "repository_id": REPOSITORY_ID,
            "file_path": rel,
            "file_name": file_info["file_name"],
            "directory": file_info["directory"],
            "extension": extension,
            "language": "javascript",
            "file_type": (
                "test"
                if re.search(
                    r"(^|[./_-])(test|spec)([./_-]|$)",
                    rel,
                    re.I
                )
                else "source"
            ),
            "file_size": file_info["size"],
            "file_sha": file_info["sha"],
            "repository_revision": REPOSITORY_REVISION,
            "ast_candidate": relative.parts[0] != "resource"
        })

    columns = [
        "file_id",
        "repository_id",
        "file_path",
        "file_name",
        "directory",
        "extension",
        "language",
        "file_type",
        "file_size",
        "file_sha",
        "repository_revision",
        "ast_candidate"
    ]

    return (
        pd.DataFrame(records, columns=columns)
        .sort_values("file_path")
        .reset_index(drop=True)
    )


# Discover repository files
files_df = discover_repository()

display(files_df.head(20))

print(f"Discovered {len(files_df):,} JavaScript files")
print(f"Total source size: {files_df['file_size'].sum():,} bytes")

,file_id,repository_id,file_path,file_name,directory,extension,language,file_type,file_size,file_sha,repository_revision
0,file_611738591214a13f,georgejaymcmc/zotero,app/assets/commandLineHandler.js,commandLineHandler.js,app/assets,.js,javascript,source,3942,e48ae4933609c47065741d73674242291dd2fc60,None
1,file_da78ee625cb55359,georgejaymcmc/zotero,app/assets/prefs.js,prefs.js,app/assets,.js,javascript,source,7322,72fc9c0cf292ca30f31813589d09412e4118d0b9,None
2,file_bc42db590d995f48,georgejaymcmc/zotero,chrome/content/scaffold/content.js,content.js,chrome/content/scaffold,.js,javascript,source,386,c042dba1cba1f7eea3bdaa54e72ed549bc673205,None
3,file_02dccdc8a7fbf869,georgejaymcmc/zotero,chrome/content/scaffold/load.js,load.js,chrome/content/scaffold,.js,javascript,source,3808,15a763aaa7234a8b5fbb99a67090250a99570882,None
4,file_767abb7cb0102665,georgejaymcmc/zotero,chrome/content/scaffold/scaffold.js,scaffold.js,chrome/content/scaffold,.js,javascript,source,71688,39701bfd8b6d32cd407f68756c037abe59f1f7ff,None
5,file_18307e89f1ddb102,georgejaymcmc/zotero,chrome/content/scaffold/scaffoldItemPreview.js,scaffoldItemPreview.js,chrome/content/scaffold,.js,javascript,source,6437,38910eb57f3753409643bf1542a0264cc860a69b,None
6,file_a6b19f7562edaee3,georgejaymcmc/zotero,chrome/content/scaffold/scaffoldItemPreviews.js,scaffoldItemPreviews.js,chrome/content/scaffold,.js,javascript,source,3920,fda4f6685eab233d4558dbf696335cd69c00a6ab,None
7,file_2f58a1107b53283e,georgejaymcmc/zotero,chrome/content/scaffold/select.js,select.js,chrome/content/scaffold,.js,javascript,source,3792,d51c244edaf869a37c9cd1dd06e21dffb4c47bfb,None
8,file_89397ac970c2c582,georgejaymcmc/zotero,chrome/content/scaffold/templates/newWeb.js,newWeb.js,chrome/content/scaffold/templates,.js,javascript,source,1931,791ec825ff8a8ec1bc7f018a84a942e98cb89625,None
9,file_eeea579b76990083,georgejaymcmc/zotero,chrome/content/scaffold/templates/scrapeBibTeX.js,scrapeBibTeX.js,chrome/content/scaffold/templates,.js,javascript,source,750,42f8d7eecf15dbe8ef46ba409d42da5f737fc2bd,None


Discovered 578 JavaScript files
Total source size: 11,262,711 bytes


###### 3. Parse `javascript` files

In [11]:
source_by_file = {}
retrieval_errors = []

for row in files_df.itertuples(index=False):

    try:
        source_by_file[row.file_path] = github.get_blob_content(
            owner,
            repo,
            row.file_sha
        )

    except Exception as e:
        retrieval_errors.append({
            "file_path": row.file_path,
            "file_sha": row.file_sha,
            "error": str(e)
        })

print(f"Retrieved: {len(source_by_file):,} files")
print(f"Errors: {len(retrieval_errors):,}")

Retrieved: 578 files
Errors: 0


In [13]:
# Validating Tree-sitter parsing
ast_trees = {}
parse_errors = []

ast_files_df = files_df[
    files_df["ast_candidate"]
].copy()

for row in ast_files_df.itertuples(index=False):

    file_path = row.file_path

    try:
        ast_trees[file_path] = parser.parse(
            source_by_file[file_path]
        )

    except Exception as e:
        parse_errors.append({
            "file_path": file_path,
            "error": str(e)
        })

print(f"AST candidates: {len(ast_files_df):,}")
print(f"Successfully parsed: {len(ast_trees):,}")
print(f"Parser exceptions: {len(parse_errors):,}")

files_with_syntax_errors = [
    file_path
    for file_path, tree in ast_trees.items()
    if tree.root_node.has_error
]

print(f"Files with syntax errors: {len(files_with_syntax_errors)}")

Successfully parsed: 578
Parse errors: 0
Files with syntax errors: 28


In [14]:
# Examine files with parsing errors
syntax_error_records = []

for file_path, tree in ast_trees.items():

    root = tree.root_node

    if root.has_error:
        error_nodes = [
            node
            for node in walk_tree(root)
            if node.type == "ERROR" or node.is_missing
        ]

        syntax_error_records.append({
            "file_path": file_path,
            "error_node_count": len(error_nodes),
            "error_types": sorted({
                node.type for node in error_nodes
            }),
            "error_locations": [
                {
                    "type": node.type,
                    **node_location(node),
                    "text": node_text(
                        node,
                        source_by_file[file_path]
                    )[:100]
                }
                for node in error_nodes[:5]
            ]
        })

syntax_errors_df = pd.DataFrame(syntax_error_records)

display(
    syntax_errors_df[
        ["file_path", "error_node_count", "error_types"]
    ]
)


,file_path,error_node_count,error_types
0,resource/classnames.js,1,[ERROR]
1,resource/jspath.js,1,[ERROR]
2,resource/monacopilot-core.mjs,2,[ERROR]
3,resource/monacopilot.mjs,1,[ERROR]
4,resource/pako.js,1,[ERROR]
5,resource/prop-types.js,1,[ERROR]
6,resource/react-autosuggest.js,1,[ERROR]
7,resource/react-dom-server.js,1,[ERROR]
8,resource/react-dom.js,1,[ERROR]
9,resource/react-intl.js,1,[ERROR]


In [15]:
# Special file handling
file_path = "test/tests/rtfScanTest.js"

print(
    source_by_file[file_path].decode(
        "utf-8",
        errors="replace"
    )[2400:3300]
)

qual(decodeRTF(String.raw`za\'bf\'f3\'b3\'e6`, "windows-1250"), "zażółć");
			assert.equal(decodeRTF(String.raw`\u913?\u952?\u942?\u957?\u945?`), "Αθήνα");
			assert.equal(decodeRTF(String.raw`\u55357?\u56832?\u55357?\u56960?`), "😀🚀");
			assert.equal(decodeRTF(String.raw`\uc0\u55357\u56832\u55357\u56960`), "😀🚀");
		});

		it('should not affect literal backslash', () => {
			assert.equal(decodeRTF(String.raw`\\'ab`), String.raw`\\'ab`);
			assert.equal(decodeRTF(String.raw`\\u913`), String.raw`\\u913`);
			assert.equal(decodeRTF(String.raw`\\uc9`), String.raw`\\uc9`);
		});

		it('should track current value of fallback characters', () => {
			assert.equal(decodeRTF(String.raw`\u913?{\uc0\u952\u942\uc2{\u957??}\u945??}`), "Α{θή{ν}α}");
		});
		
		it('should keep track of the current codepage', () => {
			// F1 is ń in windows-1250 and ñ in windows-1252
			assert.equal(decodeRTF(String.raw


###### 4. AST Inspection Utilities

In [19]:
def print_ast(node, source_code: bytes, max_depth: int = 5, depth: int = 0):
    """Readable AST outline for debugging unfamiliar syntax."""
    indent = "  " * depth
    snippet = node_text(node, source_code).replace("\\n", " ")
    if len(snippet) > 100:
        snippet = snippet[:97] + "..."
    print(f"{indent}{node.type}: {snippet!r}")
    if depth < max_depth:
        for child in node.named_children:
            print_ast(child, source_code, max_depth, depth + 1)


def child_by_type(node, node_type: str):
    return next((c for c in node.named_children if c.type == node_type), None)


def descendants_by_type(node, node_types: set[str]):
    return [n for n in walk_tree(node) if n.type in node_types]


# Inspect one file by relative path.
def inspect_file(relative_path, max_depth=4):

    match = files_df.loc[files_df.file_path == relative_path]

    if match.empty:
        raise KeyError(f"File not found in files_df: {relative_path}")

    if relative_path not in ast_trees:
        raise KeyError(f"AST not available for file: {relative_path}")

    print_ast(
        ast_trees[relative_path].root_node,
        source_by_file[relative_path],
        max_depth=max_depth
    )

# Example:
# inspect_file("chrome/content/zotero/createParentDialog.js", max_depth=4)


In [20]:
inspect_file("chrome/content/zotero/createParentDialog.js", max_depth=4)

program: '/*\n    ***** BEGIN LICENSE BLOCK *****\n\n    Copyright © 2009 Center for History and New Media\n   ...'
  comment: '/*\n    ***** BEGIN LICENSE BLOCK *****\n\n    Copyright © 2009 Center for History and New Media\n   ...'
  comment: '/* global Zotero_Lookup: false */'
  lexical_declaration: 'const ZoteroCreateParentDialog = { // eslint-disable-line no-unused-vars\n\tinit() {\n\t\tthis.io = wi...'
    variable_declarator: 'ZoteroCreateParentDialog = { // eslint-disable-line no-unused-vars\n\tinit() {\n\t\tthis.io = window.a...'
      identifier: 'ZoteroCreateParentDialog'
      object: '{ // eslint-disable-line no-unused-vars\n\tinit() {\n\t\tthis.io = window.arguments[0];\n\n\t\tthis.inputE...'
        comment: '// eslint-disable-line no-unused-vars'
        method_definition: "init() {\n\t\tthis.io = window.arguments[0];\n\n\t\tthis.inputEl = document.getElementById('parent-item-..."
        method_definition: 'async performLookup() {\n\t\tlet newItems = await Zotero_Looku

###### 5. Symbol Extraction

In [21]:
FUNCTION_TYPES = {
    "function_declaration", "generator_function_declaration",
    "function_expression", "generator_function", "arrow_function",
    "method_definition"
}
CLASS_TYPES = {"class_declaration", "class"}
VARIABLE_DECL_TYPES = {"lexical_declaration", "variable_declaration"}
SCOPE_NODE_TYPES = {
    "program", "function_declaration", "generator_function_declaration",
    "function_expression", "generator_function", "arrow_function",
    "method_definition", "statement_block", "class_declaration", "class",
    "catch_clause", "for_statement", "for_in_statement"
}


def first_identifier(node):
    if node is None:
        return None
    for n in walk_tree(node):
        if n.type in {"identifier", "property_identifier", "shorthand_property_identifier"}:
            return n
    return None


def declaration_name_node(node):
    # Function/class declarations: name is usually a direct identifier child.
    for child in node.named_children:
        if child.type == "identifier":
            return child
    return None


def extract_symbols(file_id: str, tree, source_code: bytes) -> pd.DataFrame:
    records = []
    root = tree.root_node

    for node in walk_tree(root):
        symbol_type = None
        name_node = None
        declaration_type = node.type

        if node.type in {"function_declaration", "generator_function_declaration"}:
            symbol_type = "function"
            name_node = declaration_name_node(node)
        elif node.type in {"class_declaration", "class"}:
            symbol_type = "class"
            name_node = declaration_name_node(node)
        elif node.type == "method_definition":
            symbol_type = "method"
            name_node = child_by_type(node, "property_identifier") or first_identifier(node)
        elif node.type == "variable_declarator":
            name_node = child_by_type(node, "identifier")
            if name_node is None:
                # Destructuring is retained as a declaration record, but individual
                # bound names are expanded in a later enhancement.
                name_node = child_by_type(node, "array_pattern") or child_by_type(node, "object_pattern")
            if name_node is not None:
                parent = node.parent
                kind = parent.type if parent else "variable_declarator"
                symbol_type = "variable"
                declaration_type = kind
        elif node.type == "function_expression":
            symbol_type = "function_expression"
            name_node = declaration_name_node(node)
            if name_node is None:
                name_node = node  # anonymous function; synthetic name below
        elif node.type == "arrow_function":
            symbol_type = "arrow_function"
            name_node = node  # synthetic name below

        if symbol_type is None or name_node is None:
            continue

        name = node_text(name_node, source_code).strip()
        if name_node == node:
            name = f"<anonymous_{node.start_point.row + 1}_{node.start_point.column + 1}>"

        loc = node_location(node)
        records.append({
            "symbol_id": stable_id(file_id, symbol_type, name, loc["start_byte"], prefix="sym_"),
            "file_id": file_id,
            "name": name,
            "qualified_name": None,  # populated after parent/scope analysis
            "symbol_type": symbol_type,
            "declaration_type": declaration_type,
            "parent_symbol_id": None,
            "scope_id": None,
            "is_exported": None,  # populated from exports_df
            "start_line": loc["start_line"],
            "end_line": loc["end_line"],
            "start_column": loc["start_column"],
            "end_column": loc["end_column"],
            "start_byte": loc["start_byte"],
            "end_byte": loc["end_byte"],
            "source_text": node_text(node, source_code),
        })

    return pd.DataFrame(records)


symbol_frames = [
    extract_symbols(fid, tree, source_by_file[fid])
    for fid, tree in ast_trees.items()
]
symbols_df = pd.concat(symbol_frames, ignore_index=True) if symbol_frames else pd.DataFrame()
display(symbols_df.head(30))


,symbol_id,file_id,name,qualified_name,symbol_type,declaration_type,parent_symbol_id,scope_id,is_exported,start_line,end_line,start_column,end_column,start_byte,end_byte,source_text
0,sym_eeadad4dc0560a5c,app/assets/commandLineHandler.js,"{ CommandLineOptions, TestOptions }",None,variable,lexical_declaration,None,None,None,1,1,5,127,4,126,"{ CommandLineOptions, TestOptions } = ChromeUt..."
1,sym_3b0d08eff5808038,app/assets/commandLineHandler.js,processTestOptions,None,variable,variable_declaration,None,None,None,32,32,5,31,1403,1429,processTestOptions = false
2,sym_d3cd37ca2d6ec503,app/assets/commandLineHandler.js,<anonymous_69_3>,None,function_expression,function_expression,None,None,None,69,103,3,3,2856,3935,async function () {\n\t\ttry {\n\t\t\tlet port...
3,sym_21cf878906be0255,app/assets/commandLineHandler.js,portOrPath,None,variable,lexical_declaration,None,None,None,71,71,8,93,2891,2976,portOrPath = Services.prefs.getBranch('').getI...
4,sym_0db84f83c1429247,app/assets/commandLineHandler.js,{ DevToolsLoader },None,variable,lexical_declaration,None,None,None,73,75,10,5,2991,3099,{ DevToolsLoader } = ChromeUtils.importESModul...
5,sym_281dac345cf207b2,app/assets/commandLineHandler.js,loader,None,variable,lexical_declaration,None,None,None,76,78,10,6,3110,3173,loader = new DevToolsLoader({\n\t\t\t\tfreshCo...
6,sym_962f420570f783cf,app/assets/commandLineHandler.js,{ DevToolsServer },None,variable,lexical_declaration,None,None,None,79,79,10,80,3184,3254,"{ DevToolsServer } = loader.require(""devtools/..."
7,sym_b30c6e0a7b612583,app/assets/commandLineHandler.js,{ SocketListener },None,variable,lexical_declaration,None,None,None,80,80,10,80,3265,3335,"{ SocketListener } = loader.require(""devtools/..."
8,sym_340803e89f92fd2d,app/assets/commandLineHandler.js,socketOptions,None,variable,lexical_declaration,None,None,None,90,90,10,40,3571,3601,socketOptions = { portOrPath }
9,sym_c5727e816fee1abb,app/assets/commandLineHandler.js,listener,None,variable,lexical_declaration,None,None,None,91,91,10,70,3612,3672,"listener = new SocketListener(DevToolsServer, ..."


In [23]:
# Outputs of symbols_df
print(symbols_df.head(20))
symbols_df.columns.tolist()

               symbol_id                             file_id  \
0   sym_eeadad4dc0560a5c    app/assets/commandLineHandler.js   
1   sym_3b0d08eff5808038    app/assets/commandLineHandler.js   
2   sym_d3cd37ca2d6ec503    app/assets/commandLineHandler.js   
3   sym_21cf878906be0255    app/assets/commandLineHandler.js   
4   sym_0db84f83c1429247    app/assets/commandLineHandler.js   
5   sym_281dac345cf207b2    app/assets/commandLineHandler.js   
6   sym_962f420570f783cf    app/assets/commandLineHandler.js   
7   sym_b30c6e0a7b612583    app/assets/commandLineHandler.js   
8   sym_340803e89f92fd2d    app/assets/commandLineHandler.js   
9   sym_c5727e816fee1abb    app/assets/commandLineHandler.js   
10  sym_b56ea2c2d09b48e9  chrome/content/scaffold/content.js   
11  sym_d7f11767ca70d14d  chrome/content/scaffold/content.js   
12  sym_e377dfa1f715755c     chrome/content/scaffold/load.js   
13  sym_5e4ba843bb8eb24d     chrome/content/scaffold/load.js   
14  sym_78602db967d29e44     chrome/cont

['symbol_id',
 'file_id',
 'name',
 'qualified_name',
 'symbol_type',
 'declaration_type',
 'parent_symbol_id',
 'scope_id',
 'is_exported',
 'start_line',
 'end_line',
 'start_column',
 'end_column',
 'start_byte',
 'end_byte',
 'source_text']

In [24]:
# Check distrubution of symbol extraction
display(
    symbols_df["symbol_type"]
    .value_counts()
    .to_frame("count")
)

,count
symbol_type,
variable,40542
function_expression,13940
arrow_function,4527
method,2692
function,1377
class,153


In [25]:
# Check symbols of representative file
target_file = "chrome/content/zotero/createParentDialog.js"

display(
    symbols_df.loc[
        symbols_df["file_id"] == target_file,
        [
            "name",
            "symbol_type",
            "declaration_type",
            "start_line",
            "end_line",
            "source_text"
        ]
    ].head(30)
)

,name,symbol_type,declaration_type,start_line,end_line,source_text
3242,ZoteroCreateParentDialog,variable,lexical_declaration,28,95,ZoteroCreateParentDialog = { // eslint-disable...
3243,init,method,method_definition,29,48,init() {\n\t\tthis.io = window.arguments[0];\n...
3244,performLookup,method,method_definition,50,62,async performLookup() {\n\t\tlet newItems = aw...
3245,newItems,variable,lexical_declaration,51,55,newItems = await Zotero_Lookup.addItemsFromIde...
3246,handleInput,method,method_definition,64,67,handleInput(event) {\n\t\tconst input = event....
3247,input,variable,lexical_declaration,65,65,input = event.target.value.trim()
3248,handleStatusChange,method,method_definition,69,79,handleStatusChange(isLookingUp)\t{\n\t\tthis.i...
3249,handleAcceptClick,method,method_definition,81,89,handleAcceptClick(ev) {\n\t\tev.preventDefault...
3250,handleManualEntry,method,method_definition,91,94,handleManualEntry() {\n\t\tthis.io.dataOut = {...


###### 6. Scope extraction and symbol-to-scope assignment

In [26]:
def extract_scopes(file_id: str, tree) -> tuple[pd.DataFrame, dict[int, str]]:
    """Create lexical-scope candidates and map AST node start bytes to scope IDs.

    This is an initial structural scope model. JavaScript's exact var/let/const
    binding rules are applied during resolution, not inferred solely from this tree.
    """
    root = tree.root_node
    records = []
    node_scope = {}

    def visit(node, current_scope_id=None, parent_scope_id=None):
        creates_scope = node.type in SCOPE_NODE_TYPES
        scope_id = current_scope_id

        if creates_scope:
            loc = node_location(node)
            scope_id = stable_id(file_id, node.type, loc["start_byte"], prefix="scope_")
            records.append({
                "scope_id": scope_id,
                "file_id": file_id,
                "parent_scope_id": parent_scope_id,
                "scope_type": (
                    "module" if node.type == "program" else
                    "function" if node.type in FUNCTION_TYPES else
                    "class" if node.type in CLASS_TYPES else
                    "block" if node.type == "statement_block" else
                    "catch" if node.type == "catch_clause" else
                    node.type
                ),
                "owner_symbol_id": None,
                "start_line": loc["start_line"],
                "end_line": loc["end_line"],
                "start_byte": loc["start_byte"],
                "end_byte": loc["end_byte"],
            })
            parent_for_children = scope_id
        else:
            parent_for_children = parent_scope_id

        node_scope[node.start_byte] = scope_id
        for child in node.named_children:
            visit(child, scope_id, parent_for_children)

    visit(root)
    return pd.DataFrame(records), node_scope


scope_frames = []
scope_maps = {}
for fid, tree in ast_trees.items():
    sdf, node_map = extract_scopes(fid, tree)
    scope_frames.append(sdf)
    scope_maps[fid] = node_map

scopes_df = pd.concat(scope_frames, ignore_index=True) if scope_frames else pd.DataFrame()


# Assign each symbol to the narrowest containing scope by source range.
def assign_symbol_scopes(symbols: pd.DataFrame, scopes: pd.DataFrame) -> pd.DataFrame:
    if symbols.empty or scopes.empty:
        return symbols
    result = symbols.copy()
    for idx, sym in result.iterrows():
        candidates = scopes[
            (scopes.file_id == sym.file_id) &
            (scopes.start_byte <= sym.start_byte) &
            (scopes.end_byte >= sym.end_byte)
            ].copy()
        if not candidates.empty:
            candidates["span"] = candidates.end_byte - candidates.start_byte
            chosen = candidates.sort_values("span").iloc[0]
            result.at[idx, "scope_id"] = chosen.scope_id
    return result


symbols_df = assign_symbol_scopes(symbols_df, scopes_df)
display(scopes_df.head(30))
display(symbols_df[["name", "symbol_type", "scope_id", "start_line"]].head(30))


,scope_id,file_id,parent_scope_id,scope_type,owner_symbol_id,start_line,end_line,start_byte,end_byte
0,scope_54aafa45f9b4cdc9,app/assets/commandLineHandler.js,NaN,module,None,1,105,0,3942
1,scope_fe6f655821a3ec08,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,5,7,319,354
2,scope_b07c257be1a2b99e,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,10,12,434,476
3,scope_8c90f214a6765d32,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,14,16,570,612
4,scope_09a96ddd302599c9,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,20,22,932,1014
5,scope_711571ea4929e8ff,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,28,30,1324,1397
6,scope_26951c041e1e30a3,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,33,36,1476,1540
7,scope_df92ae78ffd2c11c,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,37,39,1595,1640
8,scope_cb719a9fd91af10f,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,40,42,1698,1746
9,scope_a4f65ccc8a56fd46,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,44,66,1772,2807


,name,symbol_type,scope_id,start_line
0,"{ CommandLineOptions, TestOptions }",variable,scope_54aafa45f9b4cdc9,1
1,processTestOptions,variable,scope_54aafa45f9b4cdc9,32
2,<anonymous_69_3>,function_expression,scope_4114a771dc9a0d0c,69
3,portOrPath,variable,scope_96b025d4493a471e,71
4,{ DevToolsLoader },variable,scope_96b025d4493a471e,73
5,loader,variable,scope_96b025d4493a471e,76
6,{ DevToolsServer },variable,scope_96b025d4493a471e,79
7,{ SocketListener },variable,scope_96b025d4493a471e,80
8,socketOptions,variable,scope_96b025d4493a471e,90
9,listener,variable,scope_96b025d4493a471e,91


In [27]:
# Check function with createParentDialog.js
target_file = "chrome/content/zotero/createParentDialog.js"

display(
    symbols_df.loc[
        symbols_df["file_id"] == target_file,
        [
            "name",
            "symbol_type",
            "scope_id",
            "parent_symbol_id",
            "start_line",
            "end_line"
        ]
    ].sort_values("start_line")
)

,name,symbol_type,scope_id,parent_symbol_id,start_line,end_line
3242,ZoteroCreateParentDialog,variable,scope_e807a690d1b2f289,None,28,95
3243,init,method,scope_d9236a24d70e0404,None,29,48
3244,performLookup,method,scope_bc87c45c851b427a,None,50,62
3245,newItems,variable,scope_4b2fd8ed08e5c76d,None,51,55
3246,handleInput,method,scope_00289fee34a2c3e9,None,64,67
3247,input,variable,scope_26631897e984f0a8,None,65,65
3248,handleStatusChange,method,scope_0aefa65b6e72d731,None,69,79
3249,handleAcceptClick,method,scope_0ee1653b5718256d,None,81,89
3250,handleManualEntry,method,scope_407ab593dcbbca6a,None,91,94


In [28]:
display(scopes_df.head(20))
print(scopes_df.columns.tolist())

,scope_id,file_id,parent_scope_id,scope_type,owner_symbol_id,start_line,end_line,start_byte,end_byte
0,scope_54aafa45f9b4cdc9,app/assets/commandLineHandler.js,NaN,module,None,1,105,0,3942
1,scope_fe6f655821a3ec08,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,5,7,319,354
2,scope_b07c257be1a2b99e,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,10,12,434,476
3,scope_8c90f214a6765d32,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,14,16,570,612
4,scope_09a96ddd302599c9,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,20,22,932,1014
5,scope_711571ea4929e8ff,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,28,30,1324,1397
6,scope_26951c041e1e30a3,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,33,36,1476,1540
7,scope_df92ae78ffd2c11c,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,37,39,1595,1640
8,scope_cb719a9fd91af10f,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,40,42,1698,1746
9,scope_a4f65ccc8a56fd46,app/assets/commandLineHandler.js,scope_54aafa45f9b4cdc9,block,None,44,66,1772,2807


['scope_id', 'file_id', 'parent_scope_id', 'scope_type', 'owner_symbol_id', 'start_line', 'end_line', 'start_byte', 'end_byte']


In [29]:
target_file = "chrome/content/zotero/createParentDialog.js"

display(
    scopes_df.loc[
        scopes_df["file_id"] == target_file
    ].sort_values("start_byte")
)

,scope_id,file_id,parent_scope_id,scope_type,owner_symbol_id,start_line,end_line,start_byte,end_byte
4600,scope_e807a690d1b2f289,chrome/content/zotero/createParentDialog.js,NaN,module,None,1,96,0,2813
4601,scope_d9236a24d70e0404,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,29,48,1041,1844
4602,scope_e0e2d37d7ec268a0,chrome/content/zotero/createParentDialog.js,scope_d9236a24d70e0404,block,None,29,48,1048,1844
4603,scope_bc87c45c851b427a,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,50,62,1848,2166
4604,scope_4b2fd8ed08e5c76d,chrome/content/zotero/createParentDialog.js,scope_bc87c45c851b427a,block,None,50,62,1870,2166
4605,scope_295234ee1d9b9f67,chrome/content/zotero/createParentDialog.js,scope_4b2fd8ed08e5c76d,block,None,58,61,2093,2163
4606,scope_00289fee34a2c3e9,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,64,67,2170,2280
4607,scope_26631897e984f0a8,chrome/content/zotero/createParentDialog.js,scope_00289fee34a2c3e9,block,None,64,67,2189,2280
4608,scope_0aefa65b6e72d731,chrome/content/zotero/createParentDialog.js,scope_e807a690d1b2f289,function,None,69,79,2284,2588
4609,scope_c9997f477a053827,chrome/content/zotero/createParentDialog.js,scope_0aefa65b6e72d731,block,None,69,79,2316,2588


###### 7. Import and export extraction

In [30]:
def extract_imports(file_id: str, tree, source_code: bytes) -> pd.DataFrame:
    records = []
    for node in walk_tree(tree.root_node):
        if node.type == "import_statement":
            text = node_text(node, source_code)
            source_node = next((c for c in node.named_children if c.type == "string"), None)
            module_specifier = node_text(source_node, source_code).strip("'\\\"") if source_node else None
            records.append({
                "import_id": stable_id(file_id, node.start_byte, prefix="imp_"),
                "file_id": file_id,
                "module_specifier": module_specifier,
                "import_text": text,
                "import_kind": "es_module",
                **node_location(node),
            })
        elif node.type == "call_expression":
            fn = child_by_type(node, "identifier")
            args = child_by_type(node, "arguments")
            if fn and node_text(fn, source_code) == "require" and args:
                string_arg = next((c for c in args.named_children if c.type == "string"), None)
                spec = node_text(string_arg, source_code).strip("'\\\"") if string_arg else None
                records.append({
                    "import_id": stable_id(file_id, node.start_byte, prefix="imp_"),
                    "file_id": file_id,
                    "module_specifier": spec,
                    "import_text": node_text(node, source_code),
                    "import_kind": "commonjs_require",
                    **node_location(node),
                })
    return pd.DataFrame(records)


def extract_exports(file_id: str, tree, source_code: bytes) -> pd.DataFrame:
    records = []
    for node in walk_tree(tree.root_node):
        if node.type == "export_statement":
            text = node_text(node, source_code)
            records.append({
                "export_id": stable_id(file_id, node.start_byte, prefix="exp_"),
                "file_id": file_id,
                "export_text": text,
                "export_kind": "es_module",
                **node_location(node),
            })
        elif node.type == "assignment_expression":
            left = child_by_type(node, "member_expression")
            if left and node_text(left, source_code).replace(" ", "") == "module.exports":
                records.append({
                    "export_id": stable_id(file_id, node.start_byte, prefix="exp_"),
                    "file_id": file_id,
                    "export_text": node_text(node, source_code),
                    "export_kind": "commonjs_module_exports",
                    **node_location(node),
                })
    return pd.DataFrame(records)


imports_df = pd.concat(
    [extract_imports(fid, tree, source_by_file[fid]) for fid, tree in ast_trees.items()],
    ignore_index=True
) if ast_trees else pd.DataFrame()

exports_df = pd.concat(
    [extract_exports(fid, tree, source_by_file[fid]) for fid, tree in ast_trees.items()],
    ignore_index=True
) if ast_trees else pd.DataFrame()

display(imports_df.head(30))
display(exports_df.head(30))


,import_id,file_id,module_specifier,import_text,import_kind,start_line,start_column,end_line,end_column,start_byte,end_byte
0,imp_d158a1fb0518df49,chrome/content/scaffold/zoteroWebTranslationEn...,chrome://zotero/content/xpcom/translate/testTr...,import { AbstractWebTranslationEnvironment } f...,es_module,1,1,1,130,0,129
1,imp_e493445f19fe03e3,chrome/content/zotero/HiddenBrowser.mjs,chrome://zotero/content/BlockingObserver.mjs,"import { BlockingObserver } from ""chrome://zot...",es_module,27,1,27,81,939,1019
2,imp_cb4be66627adec70,chrome/content/zotero/HiddenBrowser.mjs,chrome://zotero/content/actors/ActorManager.mjs,"import ""chrome://zotero/content/actors/ActorMa...",es_module,28,1,28,58,1020,1077
3,imp_7069ee4ea240432e,chrome/content/zotero/ZoteroProtocolHandler.mjs,resource://gre/modules/NetUtil.sys.mjs,"import { NetUtil } from ""resource://gre/module...",es_module,35,1,35,66,1361,1426
4,imp_d1ad3d311f92135c,chrome/content/zotero/ZoteroProtocolHandler.mjs,chrome://zotero/content/zotero.mjs,"import { Zotero } from ""chrome://zotero/conten...",es_module,45,1,45,61,1676,1736
5,imp_cd2b7380c8bcdbd0,chrome/content/zotero/ace/aceWrapper.js,ace/mode/javascript,"require(""ace/mode/javascript"")",commonjs_require,37,19,37,49,1245,1275
6,imp_0931f2c723de92ab,chrome/content/zotero/ace/aceWrapper.js,ace/mode/text,"require(""ace/mode/text"")",commonjs_require,38,13,38,37,1294,1318
7,imp_d0c66847273a737a,chrome/content/zotero/ace/aceWrapper.js,ace/edit_session,"require(""ace/edit_session"")",commonjs_require,39,16,39,43,1340,1367
8,imp_687d376ff5f69a67,chrome/content/zotero/actors/ActorManager.mjs,resource://gre/modules/ActorManagerParent.sys.mjs,"import ""resource://gre/modules/ActorManagerPar...",es_module,2,1,2,60,27,86
9,imp_726a4a4a2ffc2e07,chrome/content/zotero/actors/DocumentIsReadyCh...,chrome://zotero/content/actors/actorUtils.mjs,"import { documentIsReady } from ""chrome://zote...",es_module,1,1,1,81,0,80


,export_id,file_id,export_text,export_kind,start_line,start_column,end_line,end_column,start_byte,end_byte
0,exp_109af1fe96eb42b7,chrome/content/scaffold/zoteroWebTranslationEn...,export class ZoteroWebTranslationEnvironment e...,es_module,3,1,77,2,131,2267
1,exp_c7b424b7751b6abe,chrome/content/zotero/BlockingObserver.mjs,export class BlockingObserver {\n\tshouldBlock...,es_module,30,1,88,2,1038,2457
2,exp_be267970a819465d,chrome/content/zotero/EPUB.mjs,export class EPUB {\n\t_zipReader;\n\n\t_conte...,es_module,39,1,190,2,1244,5679
3,exp_d23c8e3084e62bd8,chrome/content/zotero/HiddenBrowser.mjs,export class HiddenBrowser {\n\t/**\n\t * @par...,es_module,47,1,343,2,1583,11122
4,exp_dc4e9b1185dc78f1,chrome/content/zotero/RemoteTranslate.mjs,export class RemoteTranslate {\n\t_browser = n...,es_module,33,1,289,2,1229,8640
5,exp_7977c7f49d721d31,chrome/content/zotero/ZoteroProtocolHandler.mjs,export function ZoteroProtocolHandler() {\n\tt...,es_module,47,1,895,2,1738,25288
6,exp_3089388998dcc90c,chrome/content/zotero/actors/DocumentIsReadyCh...,export class DocumentIsReadyChild extends JSWi...,es_module,3,1,12,2,82,353
7,exp_22f1725401402091,chrome/content/zotero/actors/ExternalLinkHandl...,export class ExternalLinkHandlerChild extends ...,es_module,1,1,67,2,0,1698
8,exp_563c13b4ffc99d7c,chrome/content/zotero/actors/ExternalLinkHandl...,export class ExternalLinkHandlerParent extends...,es_module,5,1,14,2,99,302
9,exp_2eb99ac8bd41ea1c,chrome/content/zotero/actors/FeedAbstractChild...,export class FeedAbstractChild extends JSWindo...,es_module,1,1,62,2,0,1625


In [33]:
print(imports_df.columns.tolist())
display(imports_df.head(10))

['import_id', 'file_id', 'module_specifier', 'import_text', 'import_kind', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,import_id,file_id,module_specifier,import_text,import_kind,start_line,start_column,end_line,end_column,start_byte,end_byte
0,imp_d158a1fb0518df49,chrome/content/scaffold/zoteroWebTranslationEn...,chrome://zotero/content/xpcom/translate/testTr...,import { AbstractWebTranslationEnvironment } f...,es_module,1,1,1,130,0,129
1,imp_e493445f19fe03e3,chrome/content/zotero/HiddenBrowser.mjs,chrome://zotero/content/BlockingObserver.mjs,"import { BlockingObserver } from ""chrome://zot...",es_module,27,1,27,81,939,1019
2,imp_cb4be66627adec70,chrome/content/zotero/HiddenBrowser.mjs,chrome://zotero/content/actors/ActorManager.mjs,"import ""chrome://zotero/content/actors/ActorMa...",es_module,28,1,28,58,1020,1077
3,imp_7069ee4ea240432e,chrome/content/zotero/ZoteroProtocolHandler.mjs,resource://gre/modules/NetUtil.sys.mjs,"import { NetUtil } from ""resource://gre/module...",es_module,35,1,35,66,1361,1426
4,imp_d1ad3d311f92135c,chrome/content/zotero/ZoteroProtocolHandler.mjs,chrome://zotero/content/zotero.mjs,"import { Zotero } from ""chrome://zotero/conten...",es_module,45,1,45,61,1676,1736
5,imp_cd2b7380c8bcdbd0,chrome/content/zotero/ace/aceWrapper.js,ace/mode/javascript,"require(""ace/mode/javascript"")",commonjs_require,37,19,37,49,1245,1275
6,imp_0931f2c723de92ab,chrome/content/zotero/ace/aceWrapper.js,ace/mode/text,"require(""ace/mode/text"")",commonjs_require,38,13,38,37,1294,1318
7,imp_d0c66847273a737a,chrome/content/zotero/ace/aceWrapper.js,ace/edit_session,"require(""ace/edit_session"")",commonjs_require,39,16,39,43,1340,1367
8,imp_687d376ff5f69a67,chrome/content/zotero/actors/ActorManager.mjs,resource://gre/modules/ActorManagerParent.sys.mjs,"import ""resource://gre/modules/ActorManagerPar...",es_module,2,1,2,60,27,86
9,imp_726a4a4a2ffc2e07,chrome/content/zotero/actors/DocumentIsReadyCh...,chrome://zotero/content/actors/actorUtils.mjs,"import { documentIsReady } from ""chrome://zote...",es_module,1,1,1,81,0,80


In [34]:
display(
    imports_df["import_kind"]
    .value_counts()
    .to_frame("count")
)

,count
import_kind,
commonjs_require,210
es_module,117


In [36]:
display(
    imports_df.groupby("file_id")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("import_count")
)

,import_count
file_id,
test/resource/pako_inflate.js,14
js-build/watch.js,12
js-build/build.js,11
chrome/content/zotero/components/editable.jsx,8
eslint.config.mjs,8
chrome/content/zotero/itemTree.jsx,8
chrome/content/zotero/collectionTree.jsx,8
chrome/content/zotero/integration/citationExplorer.js,7
js-build/sass.js,7


In [37]:
print(exports_df.columns.tolist())
display(exports_df.head(10))

['export_id', 'file_id', 'export_text', 'export_kind', 'start_line', 'start_column', 'end_line', 'end_column', 'start_byte', 'end_byte']


,export_id,file_id,export_text,export_kind,start_line,start_column,end_line,end_column,start_byte,end_byte
0,exp_109af1fe96eb42b7,chrome/content/scaffold/zoteroWebTranslationEn...,export class ZoteroWebTranslationEnvironment e...,es_module,3,1,77,2,131,2267
1,exp_c7b424b7751b6abe,chrome/content/zotero/BlockingObserver.mjs,export class BlockingObserver {\n\tshouldBlock...,es_module,30,1,88,2,1038,2457
2,exp_be267970a819465d,chrome/content/zotero/EPUB.mjs,export class EPUB {\n\t_zipReader;\n\n\t_conte...,es_module,39,1,190,2,1244,5679
3,exp_d23c8e3084e62bd8,chrome/content/zotero/HiddenBrowser.mjs,export class HiddenBrowser {\n\t/**\n\t * @par...,es_module,47,1,343,2,1583,11122
4,exp_dc4e9b1185dc78f1,chrome/content/zotero/RemoteTranslate.mjs,export class RemoteTranslate {\n\t_browser = n...,es_module,33,1,289,2,1229,8640
5,exp_7977c7f49d721d31,chrome/content/zotero/ZoteroProtocolHandler.mjs,export function ZoteroProtocolHandler() {\n\tt...,es_module,47,1,895,2,1738,25288
6,exp_3089388998dcc90c,chrome/content/zotero/actors/DocumentIsReadyCh...,export class DocumentIsReadyChild extends JSWi...,es_module,3,1,12,2,82,353
7,exp_22f1725401402091,chrome/content/zotero/actors/ExternalLinkHandl...,export class ExternalLinkHandlerChild extends ...,es_module,1,1,67,2,0,1698
8,exp_563c13b4ffc99d7c,chrome/content/zotero/actors/ExternalLinkHandl...,export class ExternalLinkHandlerParent extends...,es_module,5,1,14,2,99,302
9,exp_2eb99ac8bd41ea1c,chrome/content/zotero/actors/FeedAbstractChild...,export class FeedAbstractChild extends JSWindo...,es_module,1,1,62,2,0,1625


In [42]:
def inspect_import_export_nodes(file_path, limit=12):
    tree = ast_trees[file_path]
    source_code = source_by_file[file_path]

    count = 0

    for node in walk_tree(tree.root_node):
        if node.type not in {
            "import_statement",
            "export_statement",
            "call_expression",
            "assignment_expression",
        }:
            continue

        text = node_text(node, source_code)

        if node.type == "call_expression":
            fn = child_by_type(node, "identifier")
            if not fn or node_text(fn, source_code) != "require":
                continue

        elif node.type == "assignment_expression":
            if not any(
                term in text
                for term in ("module.exports", "exports.")
            ):
                continue

        elif node.type not in {"import_statement", "export_statement"}:
            continue

        print(f"\n{'=' * 70}")
        print(f"TYPE: {node.type}")
        print(f"SOURCE: {text[:250]}")

        for child in node.children:
            print(
                f"  {child.type:<25} "
                f"named={child.is_named:<5} "
                f"text={node_text(child, source_code)[:100]!r}"
            )

        count += 1

        if count >= limit:
            break

In [40]:
files_df.loc[
    files_df["file_path"].str.contains(
        "zoteroWebTranslationEnvironment",
        case=False,
        na=False
    ),
    ["file_path", "file_id"]
]

,file_path,file_id
14,chrome/content/scaffold/zoteroWebTranslationEn...,file_ebe5fee72d4d11cc


In [43]:
inspect_import_export_nodes(
    "chrome/content/zotero/HiddenBrowser.mjs"
)

inspect_import_export_nodes(
    "chrome/content/scaffold/zoteroWebTranslationEnvironment.mjs"
)


TYPE: import_statement
SOURCE: import { BlockingObserver } from "chrome://zotero/content/BlockingObserver.mjs";
  import                    named=0     text='import'
  import_clause             named=1     text='{ BlockingObserver }'
  from                      named=0     text='from'
  string                    named=1     text='"chrome://zotero/content/BlockingObserver.mjs"'
  ;                         named=0     text=';'

TYPE: import_statement
SOURCE: import "chrome://zotero/content/actors/ActorManager.mjs";
  import                    named=0     text='import'
  string                    named=1     text='"chrome://zotero/content/actors/ActorManager.mjs"'
  ;                         named=0     text=';'

TYPE: export_statement
SOURCE: export class HiddenBrowser {
	/**
	 * @param {Object} options
	 * @param {Boolean} [options.allowJavaScript]
	 * @param {Object} [options.docShell] Fields to set on Browser.docShell
	 * @param {Boolean} [options.blockRemoteResources] Block all remo

In [44]:
def inspect_module_patterns(limit_per_category=2):
    categories = {
        "aliased_import": [],
        "namespace_import": [],
        "default_import": [],
        "aliased_export": [],
        "default_export": [],
        "re_export": [],
        "require": [],
        "module_exports": [],
        "exports_property": [],
    }

    for file_id, tree in ast_trees.items():
        source_code = source_by_file[file_id]

        for node in walk_tree(tree.root_node):
            source = node_text(node, source_code)

            if node.type == "import_statement":
                if " as " in source and "*" in source:
                    categories["namespace_import"].append((file_id, node))
                elif " as " in source:
                    categories["aliased_import"].append((file_id, node))
                elif "import " in source and " from " in source:
                    clause = next(
                        (c for c in node.named_children
                         if c.type == "import_clause"),
                        None
                    )
                    if clause and clause.named_children:
                        first = clause.named_children[0]
                        if first.type == "identifier":
                            categories["default_import"].append((file_id, node))

            elif node.type == "export_statement":
                if "export default" in source:
                    categories["default_export"].append((file_id, node))
                elif " from " in source:
                    categories["re_export"].append((file_id, node))
                elif " as " in source:
                    categories["aliased_export"].append((file_id, node))

            elif node.type == "call_expression":
                fn = child_by_type(node, "identifier")
                if fn and node_text(fn, source_code) == "require":
                    categories["require"].append((file_id, node))

            elif node.type == "assignment_expression":
                left = child_by_type(node, "member_expression")
                if left:
                    left_text = node_text(left, source_code).replace(" ", "")
                    if left_text.startswith("module.exports"):
                        categories["module_exports"].append((file_id, node))
                    elif left_text.startswith("exports."):
                        categories["exports_property"].append((file_id, node))

    for category, matches in categories.items():
        print(f"\n{'=' * 75}")
        print(f"{category}: {len(matches)} candidates")

        for file_id, node in matches[:limit_per_category]:
            source_code = source_by_file[file_id]

            print(f"\nFILE: {file_id}")
            print(f"SOURCE: {node_text(node, source_code)[:300]!r}")

            for child in node.children:
                print(
                    f"  {child.type:<25} "
                    f"named={child.is_named!s:<5} "
                    f"text={node_text(child, source_code)[:120]!r}"
                )

                for grandchild in child.named_children:
                    print(
                        f"    {grandchild.type:<23} "
                        f"text={node_text(grandchild, source_code)[:100]!r}"
                    )


inspect_module_patterns()


aliased_import: 3 candidates

FILE: js-build/ftl-to-json.mjs
SOURCE: "import { ftlFileBaseNames as sourceFileBaseNames } from './config.js';"
  import                    named=False text='import'
  import_clause             named=True  text='{ ftlFileBaseNames as sourceFileBaseNames }'
    named_imports           text='{ ftlFileBaseNames as sourceFileBaseNames }'
  from                      named=False text='from'
  string                    named=True  text="'./config.js'"
    string_fragment         text='./config.js'
  ;                         named=False text=';'

FILE: js-build/localize-ftl.mjs
SOURCE: "import { ftlFileBaseNames as sourceFileBaseNames } from './config.js';"
  import                    named=False text='import'
  import_clause             named=True  text='{ ftlFileBaseNames as sourceFileBaseNames }'
    named_imports           text='{ ftlFileBaseNames as sourceFileBaseNames }'
  from                      named=False text='from'
  string                    named=

In [45]:
def print_named_children(node, source_code, depth=0, max_depth=4):
    if depth > max_depth:
        return

    indent = "  " * depth

    print(
        f"{indent}{node.type:<25} "
        f"{node_text(node, source_code)[:120]!r}"
    )

    for child in node.named_children:
        print_named_children(
            child,
            source_code,
            depth=depth + 1,
            max_depth=max_depth
        )


def inspect_selected_patterns():
    inspected = {
        "aliased_import": False,
        "namespace_import": False,
        "default_export": False,
        "re_export": False,
    }

    for file_id, tree in ast_trees.items():
        source_code = source_by_file[file_id]

        for node in walk_tree(tree.root_node):
            if node.type == "import_statement":
                clause = next(
                    (c for c in node.named_children
                     if c.type == "import_clause"),
                    None
                )

                if clause:
                    clause_text = node_text(clause, source_code)

                    if " as " in clause_text and not inspected["aliased_import"]:
                        category = "aliased_import"
                    elif "*" in clause_text and not inspected["namespace_import"]:
                        category = "namespace_import"
                    else:
                        continue

                    inspected[category] = True

                    print(f"\n{'=' * 70}")
                    print(f"{category}: {file_id}")
                    print_named_children(node, source_code)

            elif node.type == "export_statement":
                children = node.named_children
                child_types = [c.type for c in children]

                if (
                    "default" in node_text(node, source_code)[:30]
                    and not inspected["default_export"]
                ):
                    category = "default_export"

                elif (
                    any(c.type == "string" for c in children)
                    and not inspected["re_export"]
                ):
                    category = "re_export"

                else:
                    continue

                inspected[category] = True

                print(f"\n{'=' * 70}")
                print(f"{category}: {file_id}")
                print_named_children(node, source_code)

            if all(inspected.values()):
                return


inspect_selected_patterns()


default_export: chrome/content/zotero/components/editable.jsx
export_statement          'export default React.forwardRef((props, ref) => <Editable\n\tinnerRef={ref} {...props}\n/>);'
  call_expression           'React.forwardRef((props, ref) => <Editable\n\tinnerRef={ref} {...props}\n/>)'
    member_expression         'React.forwardRef'
      identifier                'React'
      property_identifier       'forwardRef'
    arguments                 '((props, ref) => <Editable\n\tinnerRef={ref} {...props}\n/>)'
      arrow_function            '(props, ref) => <Editable\n\tinnerRef={ref} {...props}\n/>'
        formal_parameters         '(props, ref)'
        jsx_self_closing_element  '<Editable\n\tinnerRef={ref} {...props}\n/>'

aliased_import: js-build/ftl-to-json.mjs
import_statement          "import { ftlFileBaseNames as sourceFileBaseNames } from './config.js';"
  import_clause             '{ ftlFileBaseNames as sourceFileBaseNames }'
    named_imports             '{ ftlFileBaseNa

In [46]:
def inspect_reexport(limit=3):
    count = 0

    for file_id, tree in ast_trees.items():
        source_code = source_by_file[file_id]

        for node in walk_tree(tree.root_node):
            if node.type != "export_statement":
                continue

            # Re-exports have a source module string.
            strings = [
                child for child in walk_tree(node)
                if child.type == "string"
            ]

            if not strings:
                continue

            print(f"\nFILE: {file_id}")
            print(f"SOURCE: {node_text(node, source_code)[:300]!r}")
            print_named_children(node, source_code)

            count += 1
            if count >= limit:
                return

    if count == 0:
        print("No re-export statements found.")


inspect_reexport()


FILE: chrome/content/scaffold/zoteroWebTranslationEnvironment.mjs
SOURCE: "export class ZoteroWebTranslationEnvironment extends AbstractWebTranslationEnvironment {\n\n\t/**\n\t * @param {string} url\n\t * @param {TranslatorTester} tester\n\t * @returns {Promise<HiddenBrowser>}\n\t */\n\tasync fetchPage(url, { tester }) {\n\t\tconst { HiddenBrowser } = ChromeUtils.importESModule('chrome://"
export_statement          'export class ZoteroWebTranslationEnvironment extends AbstractWebTranslationEnvironment {\n\n\t/**\n\t * @param {string} url\n\t'
  class_declaration         'class ZoteroWebTranslationEnvironment extends AbstractWebTranslationEnvironment {\n\n\t/**\n\t * @param {string} url\n\t * @par'
    identifier                'ZoteroWebTranslationEnvironment'
    class_heritage            'extends AbstractWebTranslationEnvironment'
      identifier                'AbstractWebTranslationEnvironment'
    class_body                '{\n\n\t/**\n\t * @param {string} url\n\t * @param {Tran

In [47]:
def extract_imports_with_bindings(file_id, tree, source_code):
    import_records = []
    binding_records = []

    for node in walk_tree(tree.root_node):
        if node.type != "import_statement":
            continue

        import_id = stable_id(file_id, node.start_byte, prefix="imp_")

        source_node = next(
            (c for c in node.named_children if c.type == "string"),
            None
        )

        module_specifier = (
            node_text(source_node, source_code)[1:-1]
            if source_node else None
        )

        clause = next(
            (c for c in node.named_children
             if c.type == "import_clause"),
            None
        )

        import_records.append({
            "import_id": import_id,
            "file_id": file_id,
            "module_specifier": module_specifier,
            "import_text": node_text(node, source_code),
            "import_kind": "es_module",
            **node_location(node),
        })

        if clause is None:
            # Side-effect import: import "module";
            binding_records.append({
                "import_binding_id": stable_id(
                    import_id, "side_effect", prefix="imb_"
                ),
                "import_id": import_id,
                "imported_name": None,
                "local_name": None,
                "binding_kind": "side_effect",
                "is_aliased": False,
            })
            continue

        for child in clause.named_children:

            if child.type == "identifier":
                # Default import
                local_name = node_text(child, source_code)

                binding_records.append({
                    "import_binding_id": stable_id(
                        import_id, "default", local_name, prefix="imb_"
                    ),
                    "import_id": import_id,
                    "imported_name": "default",
                    "local_name": local_name,
                    "binding_kind": "default",
                    "is_aliased": False,
                })

            elif child.type == "namespace_import":
                # import * as localName from "module";
                identifier = next(
                    (c for c in child.named_children
                     if c.type == "identifier"),
                    None
                )

                local_name = (
                    node_text(identifier, source_code)
                    if identifier else None
                )

                binding_records.append({
                    "import_binding_id": stable_id(
                        import_id, "namespace", local_name, prefix="imb_"
                    ),
                    "import_id": import_id,
                    "imported_name": "*",
                    "local_name": local_name,
                    "binding_kind": "namespace",
                    "is_aliased": False,
                })

            elif child.type == "named_imports":
                for specifier in child.named_children:
                    if specifier.type != "import_specifier":
                        continue

                    identifiers = [
                        c for c in specifier.named_children
                        if c.type == "identifier"
                    ]

                    if not identifiers:
                        continue

                    imported_name = node_text(
                        identifiers[0], source_code
                    )

                    local_name = (
                        node_text(identifiers[1], source_code)
                        if len(identifiers) > 1
                        else imported_name
                    )

                    binding_records.append({
                        "import_binding_id": stable_id(
                            import_id,
                            imported_name,
                            local_name,
                            prefix="imb_"
                        ),
                        "import_id": import_id,
                        "imported_name": imported_name,
                        "local_name": local_name,
                        "binding_kind": "named",
                        "is_aliased": imported_name != local_name,
                    })

    return (
        pd.DataFrame(import_records),
        pd.DataFrame(binding_records)
    )

In [48]:
import_results = [
    extract_imports_with_bindings(
        file_id,
        tree,
        source_by_file[file_id]
    )
    for file_id, tree in ast_trees.items()
]

imports_df = pd.concat(
    [result[0] for result in import_results],
    ignore_index=True
)

import_bindings_df = pd.concat(
    [result[1] for result in import_results],
    ignore_index=True
)

In [49]:
print("Import statements:", len(imports_df))
print("Import bindings:", len(import_bindings_df))

display(
    import_bindings_df["binding_kind"]
    .value_counts()
    .to_frame("count")
)

display(
    import_bindings_df[
        import_bindings_df["is_aliased"]
    ]
)

display(
    imports_df[
        imports_df["module_specifier"] == "./config.js"
    ][["file_id", "module_specifier", "import_text"]]
)

Import statements: 117
Import bindings: 147


,count
binding_kind,
named,85
default,60
side_effect,2


,import_binding_id,import_id,imported_name,local_name,binding_kind,is_aliased
128,imb_c9023725aafa3a50,imp_b925aab3f58a7169,ftlFileBaseNames,sourceFileBaseNames,named,True
138,imb_935cd3be12e27df3,imp_57ba3cae38c39042,ftlFileBaseNames,sourceFileBaseNames,named,True
142,imb_fb019b065593cc04,imp_0c6dc5c005f1687c,BluebirdShimPromise,Promise,named,True


,file_id,module_specifier,import_text
104,js-build/ftl-to-json.mjs,./config.js,import { ftlFileBaseNames as sourceFileBaseNam...
110,js-build/localize-ftl.mjs,./config.js,import { ftlFileBaseNames as sourceFileBaseNam...


In [50]:
display(
    import_bindings_df["binding_kind"]
    .value_counts()
    .to_frame("count")
)

display(
    import_bindings_df[
        import_bindings_df["is_aliased"]
    ][
        ["import_id", "imported_name", "local_name", "binding_kind"]
    ]
)

display(
    imports_df[
        imports_df["module_specifier"] == "./config.js"
    ][["file_id", "module_specifier", "import_text"]]
)

,count
binding_kind,
named,85
default,60
side_effect,2


,import_id,imported_name,local_name,binding_kind
128,imp_b925aab3f58a7169,ftlFileBaseNames,sourceFileBaseNames,named
138,imp_57ba3cae38c39042,ftlFileBaseNames,sourceFileBaseNames,named
142,imp_0c6dc5c005f1687c,BluebirdShimPromise,Promise,named


,file_id,module_specifier,import_text
104,js-build/ftl-to-json.mjs,./config.js,import { ftlFileBaseNames as sourceFileBaseNam...
110,js-build/localize-ftl.mjs,./config.js,import { ftlFileBaseNames as sourceFileBaseNam...


In [51]:
commonjs_records = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "call_expression":
            continue

        fn = child_by_type(node, "identifier")
        args = child_by_type(node, "arguments")

        if fn and node_text(fn, source_code) == "require" and args:
            string_arg = next(
                (c for c in args.named_children if c.type == "string"),
                None
            )

            spec = (
                node_text(string_arg, source_code).strip("'\"")
                if string_arg else None
            )

            commonjs_records.append({
                "file_id": file_path,
                "module_specifier": spec,
                "import_text": node_text(node, source_code),
                "start_line": node.start_point.row + 1,
                "start_byte": node.start_byte,
            })

commonjs_imports_df = pd.DataFrame(commonjs_records)

print("CommonJS require calls:", len(commonjs_imports_df))

display(
    commonjs_imports_df[
        ["file_id", "module_specifier", "import_text"]
    ].head(20)
)

CommonJS require calls: 210


,file_id,module_specifier,import_text
0,chrome/content/zotero/ace/aceWrapper.js,ace/mode/javascript,"require(""ace/mode/javascript"")"
1,chrome/content/zotero/ace/aceWrapper.js,ace/mode/text,"require(""ace/mode/text"")"
2,chrome/content/zotero/ace/aceWrapper.js,ace/edit_session,"require(""ace/edit_session"")"
3,chrome/content/zotero/collectionTree.jsx,react,require('react')
4,chrome/content/zotero/collectionTree.jsx,react-dom,require('react-dom')
5,chrome/content/zotero/collectionTree.jsx,./libraryTree,require('./libraryTree')
6,chrome/content/zotero/collectionTree.jsx,components/virtualized-table,require('components/virtualized-table')
7,chrome/content/zotero/collectionTree.jsx,components/icons,require('components/icons')
8,chrome/content/zotero/collectionTree.jsx,components/utils,require('components/utils')
9,chrome/content/zotero/collectionTree.jsx,./components/utils,"require(""./components/utils"")"


In [52]:
require_patterns = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "call_expression":
            continue

        fn = child_by_type(node, "identifier")

        if not fn or node_text(fn, source_code) != "require":
            continue

        parent = node.parent

        require_patterns.append({
            "file_id": file_path,
            "parent_type": parent.type if parent else None,
            "require_text": node_text(node, source_code),
            "parent_text": (
                node_text(parent, source_code)
                if parent else None
            ),
        })

require_patterns_df = pd.DataFrame(require_patterns)

display(
    require_patterns_df["parent_type"]
    .value_counts()
    .to_frame("count")
)

display(
    require_patterns_df[
        ["parent_type", "parent_text"]
    ].head(25)
)

,count
parent_type,
variable_declarator,199
member_expression,8
return_statement,2
assignment_expression,1


,parent_type,parent_text
0,member_expression,"require(""ace/mode/javascript"").Mode"
1,member_expression,"require(""ace/mode/text"").Mode"
2,member_expression,"require(""ace/edit_session"").EditSession"
3,variable_declarator,React = require('react')
4,variable_declarator,ReactDOM = require('react-dom')
5,variable_declarator,LibraryTree = require('./libraryTree')
6,member_expression,require('components/virtualized-table').Virtua...
7,variable_declarator,{ getCSSIcon } = require('components/icons')
8,variable_declarator,{ getDragTargetOrient } = require('components/...
9,variable_declarator,"{ noop } = require(""./components/utils"")"


In [53]:
def inspect_require_pattern(parent_type, limit=3):
    count = 0

    for file_path, tree in ast_trees.items():
        source_code = source_by_file[file_path]

        for node in walk_tree(tree.root_node):
            if node.type != "call_expression":
                continue

            fn = child_by_type(node, "identifier")

            if not fn or node_text(fn, source_code) != "require":
                continue

            parent = node.parent

            if parent and parent.type == parent_type:
                print(f"\nFILE: {file_path}")
                print("SOURCE:", node_text(parent, source_code))

                print_ast(
                    parent,
                    source_code,
                    max_depth=5
                )

                count += 1

                if count >= limit:
                    return

for pattern in [
    "variable_declarator",
    "member_expression",
    "return_statement",
    "assignment_expression",
]:
    print(f"\n{'=' * 70}")
    print(pattern)
    inspect_require_pattern(pattern)


variable_declarator

FILE: chrome/content/zotero/collectionTree.jsx
SOURCE: React = require('react')
variable_declarator: "React = require('react')"
  identifier: 'React'
  call_expression: "require('react')"
    identifier: 'require'
    arguments: "('react')"
      string: "'react'"
        string_fragment: 'react'

FILE: chrome/content/zotero/collectionTree.jsx
SOURCE: ReactDOM = require('react-dom')
variable_declarator: "ReactDOM = require('react-dom')"
  identifier: 'ReactDOM'
  call_expression: "require('react-dom')"
    identifier: 'require'
    arguments: "('react-dom')"
      string: "'react-dom'"
        string_fragment: 'react-dom'

FILE: chrome/content/zotero/collectionTree.jsx
SOURCE: LibraryTree = require('./libraryTree')
variable_declarator: "LibraryTree = require('./libraryTree')"
  identifier: 'LibraryTree'
  call_expression: "require('./libraryTree')"
    identifier: 'require'
    arguments: "('./libraryTree')"
      string: "'./libraryTree'"
        string_fragment:

In [54]:
def inspect_require_destructuring(limit=5):
    count = 0

    for file_path, tree in ast_trees.items():
        source_code = source_by_file[file_path]

        for node in walk_tree(tree.root_node):
            if node.type != "variable_declarator":
                continue

            require_node = next(
                (
                    child for child in node.named_children
                    if child.type == "call_expression"
                    and child_by_type(child, "identifier")
                    and node_text(
                        child_by_type(child, "identifier"),
                        source_code
                    ) == "require"
                ),
                None
            )

            if require_node is None:
                continue

            pattern = node.named_children[0]

            if pattern.type in {
                "object_pattern",
                "array_pattern"
            }:
                print(f"\nFILE: {file_path}")
                print("SOURCE:", node_text(node, source_code))

                print_ast(
                    node,
                    source_code,
                    max_depth=6
                )

                count += 1

                if count >= limit:
                    return

inspect_require_destructuring()


FILE: chrome/content/zotero/collectionTree.jsx
SOURCE: { getCSSIcon } = require('components/icons')
variable_declarator: "{ getCSSIcon } = require('components/icons')"
  object_pattern: '{ getCSSIcon }'
    shorthand_property_identifier_pattern: 'getCSSIcon'
  call_expression: "require('components/icons')"
    identifier: 'require'
    arguments: "('components/icons')"
      string: "'components/icons'"
        string_fragment: 'components/icons'

FILE: chrome/content/zotero/collectionTree.jsx
SOURCE: { getDragTargetOrient } = require('components/utils')
variable_declarator: "{ getDragTargetOrient } = require('components/utils')"
  object_pattern: '{ getDragTargetOrient }'
    shorthand_property_identifier_pattern: 'getDragTargetOrient'
  call_expression: "require('components/utils')"
    identifier: 'require'
    arguments: "('components/utils')"
      string: "'components/utils'"
        string_fragment: 'components/utils'

FILE: chrome/content/zotero/collectionTree.jsx
SOURCE: { no

In [55]:
def inspect_aliased_destructuring(limit=5):
    count = 0

    for file_path, tree in ast_trees.items():
        source_code = source_by_file[file_path]

        for node in walk_tree(tree.root_node):
            if node.type != "variable_declarator":
                continue

            children = node.named_children

            if len(children) < 2:
                continue

            pattern = children[0]
            value = children[1]

            if pattern.type != "object_pattern":
                continue

            if value.type != "call_expression":
                continue

            fn = child_by_type(value, "identifier")

            if not fn or node_text(fn, source_code) != "require":
                continue

            if not any(
                child.type == "pair_pattern"
                for child in pattern.named_children
            ):
                continue

            print(f"\nFILE: {file_path}")
            print("SOURCE:", node_text(node, source_code))

            print_ast(
                node,
                source_code,
                max_depth=6
            )

            count += 1

            if count >= limit:
                return

inspect_aliased_destructuring()


FILE: chrome/content/zotero/itemTreeRow.js
SOURCE: { renderCell: baseRenderCell } = require('components/virtualized-table')
variable_declarator: "{ renderCell: baseRenderCell } = require('components/virtualized-table')"
  object_pattern: '{ renderCell: baseRenderCell }'
    pair_pattern: 'renderCell: baseRenderCell'
      property_identifier: 'renderCell'
      identifier: 'baseRenderCell'
  call_expression: "require('components/virtualized-table')"
    identifier: 'require'
    arguments: "('components/virtualized-table')"
      string: "'components/virtualized-table'"
        string_fragment: 'components/virtualized-table'


In [56]:
def extract_imports_with_bindings(file_id, tree, source_code):
    import_records = []
    binding_records = []

    for node in walk_tree(tree.root_node):
        if node.type != "import_statement":
            continue

        import_id = stable_id(file_id, node.start_byte, prefix="imp_")

        source_node = next(
            (c for c in node.named_children if c.type == "string"),
            None
        )

        module_specifier = (
            node_text(source_node, source_code)[1:-1]
            if source_node else None
        )

        clause = next(
            (c for c in node.named_children
             if c.type == "import_clause"),
            None
        )

        import_records.append({
            "import_id": import_id,
            "file_id": file_id,
            "module_specifier": module_specifier,
            "import_text": node_text(node, source_code),
            "import_kind": "es_module",
            **node_location(node),
        })

        if clause is None:
            # Side-effect import: import "module";
            binding_records.append({
                "import_binding_id": stable_id(
                    import_id, "side_effect", prefix="imb_"
                ),
                "import_id": import_id,
                "imported_name": None,
                "local_name": None,
                "binding_kind": "side_effect",
                "is_aliased": False,
            })
            continue

        for child in clause.named_children:

            if child.type == "identifier":
                # Default import
                local_name = node_text(child, source_code)

                binding_records.append({
                    "import_binding_id": stable_id(
                        import_id, "default", local_name, prefix="imb_"
                    ),
                    "import_id": import_id,
                    "imported_name": "default",
                    "local_name": local_name,
                    "binding_kind": "default",
                    "is_aliased": False,
                })

            elif child.type == "namespace_import":
                # import * as localName from "module";
                identifier = next(
                    (c for c in child.named_children
                     if c.type == "identifier"),
                    None
                )

                local_name = (
                    node_text(identifier, source_code)
                    if identifier else None
                )

                binding_records.append({
                    "import_binding_id": stable_id(
                        import_id, "namespace", local_name, prefix="imb_"
                    ),
                    "import_id": import_id,
                    "imported_name": "*",
                    "local_name": local_name,
                    "binding_kind": "namespace",
                    "is_aliased": False,
                })

            elif child.type == "named_imports":
                for specifier in child.named_children:
                    if specifier.type != "import_specifier":
                        continue

                    identifiers = [
                        c for c in specifier.named_children
                        if c.type == "identifier"
                    ]

                    if not identifiers:
                        continue

                    imported_name = node_text(
                        identifiers[0], source_code
                    )

                    local_name = (
                        node_text(identifiers[1], source_code)
                        if len(identifiers) > 1
                        else imported_name
                    )

                    binding_records.append({
                        "import_binding_id": stable_id(
                            import_id,
                            imported_name,
                            local_name,
                            prefix="imb_"
                        ),
                        "import_id": import_id,
                        "imported_name": imported_name,
                        "local_name": local_name,
                        "binding_kind": "named",
                        "is_aliased": imported_name != local_name,
                    })

    return (
        pd.DataFrame(import_records),
        pd.DataFrame(binding_records)
    )


def extract_commonjs_imports_with_bindings(file_id, tree, source_code):
    import_records = []
    binding_records = []

    def is_require_call(node):
        if node.type != "call_expression":
            return False

        fn = child_by_type(node, "identifier")
        return (
            fn is not None
            and node_text(fn, source_code) == "require"
        )

    def get_module_specifier(node):
        args = child_by_type(node, "arguments")

        if args is None:
            return None

        string_arg = next(
            (c for c in args.named_children if c.type == "string"),
            None
        )

        return (
            node_text(string_arg, source_code)[1:-1]
            if string_arg else None
        )

    def add_binding(import_id, imported_name, local_name,
                    binding_kind, is_aliased=False):
        binding_records.append({
            "import_binding_id": stable_id(
                import_id,
                imported_name,
                local_name,
                binding_kind,
                prefix="imb_"
            ),
            "import_id": import_id,
            "imported_name": imported_name,
            "local_name": local_name,
            "binding_kind": binding_kind,
            "is_aliased": is_aliased,
        })

    for node in walk_tree(tree.root_node):
        if not is_require_call(node):
            continue

        parent = node.parent
        module_specifier = get_module_specifier(node)

        import_id = stable_id(
            file_id, node.start_byte, prefix="imp_"
        )

        import_records.append({
            "import_id": import_id,
            "file_id": file_id,
            "module_specifier": module_specifier,
            "import_text": node_text(node, source_code),
            "import_kind": "commonjs_require",
            **node_location(node),
        })

        # Case 1: const foo = require("module")
        if parent and parent.type == "variable_declarator":
            pattern = parent.named_children[0]

            if pattern.type == "identifier":
                local_name = node_text(pattern, source_code)

                add_binding(
                    import_id,
                    "*",
                    local_name,
                    "module"
                )

            # Case 2: const { foo } = require("module")
            elif pattern.type == "object_pattern":
                for item in pattern.named_children:

                    if item.type == "shorthand_property_identifier_pattern":
                        imported_name = node_text(item, source_code)

                        add_binding(
                            import_id,
                            imported_name,
                            imported_name,
                            "named"
                        )

                    # Case 3: const { foo: bar } = require("module")
                    elif item.type == "pair_pattern":
                        property_node = next(
                            (
                                c for c in item.named_children
                                if c.type == "property_identifier"
                            ),
                            None
                        )

                        local_node = next(
                            (
                                c for c in item.named_children
                                if c.type == "identifier"
                            ),
                            None
                        )

                        if property_node and local_node:
                            imported_name = node_text(
                                property_node, source_code
                            )
                            local_name = node_text(
                                local_node, source_code
                            )

                            add_binding(
                                import_id,
                                imported_name,
                                local_name,
                                "named",
                                imported_name != local_name
                            )

        # Case 4: require("module").property
        elif parent and parent.type == "member_expression":
            property_node = next(
                (
                    c for c in parent.named_children
                    if c.type == "property_identifier"
                ),
                None
            )

            if property_node:
                imported_name = node_text(
                    property_node, source_code
                )

                add_binding(
                    import_id,
                    imported_name,
                    None,
                    "member"
                )

        # Case 5: require("module") returned directly
        elif parent and parent.type == "return_statement":
            add_binding(
                import_id,
                "*",
                None,
                "unbound"
            )

        # Case 6: _module = require("module")
        elif parent and parent.type == "assignment_expression":
            left = parent.named_children[0]

            local_name = (
                node_text(left, source_code)
                if left.type == "identifier"
                else None
            )

            add_binding(
                import_id,
                "*",
                local_name,
                "module"
            )

    return (
        pd.DataFrame(import_records),
        pd.DataFrame(binding_records)
    )


In [57]:
commonjs_results = [
    extract_commonjs_imports_with_bindings(
        file_path,
        tree,
        source_by_file[file_path]
    )
    for file_path, tree in ast_trees.items()
]

commonjs_imports_df = pd.concat(
    [result[0] for result in commonjs_results],
    ignore_index=True
)

commonjs_bindings_df = pd.concat(
    [result[1] for result in commonjs_results],
    ignore_index=True
)

print("CommonJS import statements:", len(commonjs_imports_df))
print("CommonJS bindings:", len(commonjs_bindings_df))

CommonJS import statements: 210
CommonJS bindings: 319


In [58]:
display(
    commonjs_bindings_df["binding_kind"]
    .value_counts()
    .to_frame("count")
)

display(
    commonjs_bindings_df[
        commonjs_bindings_df["is_aliased"]
    ]
)

display(
    commonjs_bindings_df[
        commonjs_bindings_df["imported_name"] == "renderCell"
    ]
)

,count
binding_kind,
named,168
module,141
member,8
unbound,2


,import_binding_id,import_id,imported_name,local_name,binding_kind,is_aliased
88,imb_1b2eca42479461bf,imp_d4c441a87a85e5e6,renderCell,baseRenderCell,named,True


,import_binding_id,import_id,imported_name,local_name,binding_kind,is_aliased
88,imb_1b2eca42479461bf,imp_d4c441a87a85e5e6,renderCell,baseRenderCell,named,True


In [59]:
duplicate_bindings = commonjs_bindings_df[
    commonjs_bindings_df.duplicated(
        subset=["import_binding_id"],
        keep=False
    )
].sort_values("import_binding_id")

print("Duplicate binding IDs:", len(duplicate_bindings))

display(duplicate_bindings)

Duplicate binding IDs: 0


,import_binding_id,import_id,imported_name,local_name,binding_kind,is_aliased


In [60]:
print(
    "Duplicate CommonJS import IDs:",
    commonjs_imports_df["import_id"].duplicated().sum()
)

Duplicate CommonJS import IDs: 0


In [61]:
imports_all_df = pd.concat(
    [imports_df, commonjs_imports_df],
    ignore_index=True
)

import_bindings_all_df = pd.concat(
    [import_bindings_df, commonjs_bindings_df],
    ignore_index=True
)

print("Import statements:", len(imports_all_df))
print("Import bindings:", len(import_bindings_all_df))

print(
    "Duplicate import IDs:",
    imports_all_df["import_id"].duplicated().sum()
)

print(
    "Duplicate binding IDs:",
    import_bindings_all_df["import_binding_id"].duplicated().sum()
)

Import statements: 327
Import bindings: 466
Duplicate import IDs: 0
Duplicate binding IDs: 0


In [62]:
print("Original export statements:", len(exports_df))

display(
    exports_df["export_kind"]
    .value_counts()
    .to_frame("count")
)

display(
    exports_df[
        ["file_id", "export_kind", "export_text"]
    ].head(20)
)

Original export statements: 142


,count
export_kind,
es_module,96
commonjs_module_exports,46


,file_id,export_kind,export_text
0,chrome/content/scaffold/zoteroWebTranslationEn...,es_module,export class ZoteroWebTranslationEnvironment e...
1,chrome/content/zotero/BlockingObserver.mjs,es_module,export class BlockingObserver {\n\tshouldBlock...
2,chrome/content/zotero/EPUB.mjs,es_module,export class EPUB {\n\t_zipReader;\n\n\t_conte...
3,chrome/content/zotero/HiddenBrowser.mjs,es_module,export class HiddenBrowser {\n\t/**\n\t * @par...
4,chrome/content/zotero/RemoteTranslate.mjs,es_module,export class RemoteTranslate {\n\t_browser = n...
5,chrome/content/zotero/ZoteroProtocolHandler.mjs,es_module,export function ZoteroProtocolHandler() {\n\tt...
6,chrome/content/zotero/actors/DocumentIsReadyCh...,es_module,export class DocumentIsReadyChild extends JSWi...
7,chrome/content/zotero/actors/ExternalLinkHandl...,es_module,export class ExternalLinkHandlerChild extends ...
8,chrome/content/zotero/actors/ExternalLinkHandl...,es_module,export class ExternalLinkHandlerParent extends...
9,chrome/content/zotero/actors/FeedAbstractChild...,es_module,export class FeedAbstractChild extends JSWindo...


In [65]:
es_export_patterns = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "export_statement":
            continue

        children = node.named_children

        es_export_patterns.append({
            "file_id": file_path,
            "child_types": ", ".join(c.type for c in children),
            "export_text": node_text(node, source_code),
        })

es_export_patterns_df = pd.DataFrame(es_export_patterns)

print("ES module export statements:", len(es_export_patterns_df))

display(
    es_export_patterns_df["child_types"]
    .value_counts()
    .to_frame("count")
)

ES module export statements: 96


,count
child_types,
class_declaration,33
function_declaration,32
lexical_declaration,9
variable_declaration,8
call_expression,5
export_clause,5
identifier,3
array,1


In [67]:
non_declaration_exports_df = es_export_patterns_df[
    ~es_export_patterns_df["child_types"].isin([
        "class_declaration",
        "function_declaration"
    ])
]

display(
    non_declaration_exports_df[
        ["file_id", "child_types", "export_text"]
    ].to_string(index=False)
)

'                                                 file_id          child_types                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [68]:
display(
    es_export_patterns_df[
        es_export_patterns_df["child_types"].str.contains("export_clause")
    ][["file_id", "export_text"]].to_string(index=False)
)

'                                                file_id                                                                                                                   export_text\n            chrome/content/zotero/components/tooltip.js                                                                                                    export { Zotero_Tooltip };\n              chrome/content/zotero/components/utils.js export {\\n\\tnextHTMLID, noop, getDragTargetOrient, createDragHandler, scrollIntoViewIfNeeded, stopPropagation, pick, omit\\n};\n                 chrome/content/zotero/import/citavi.js                                                                                            export { ImportCitaviAnnotatons };\n             chrome/content/zotero/modules/immutable.js                                                                                              export { removeKeys, pickKeys };\nchrome/content/zotero/xpcom/pluginAPI/pluginAPIBase.mjs                          

In [69]:
file_path = "chrome/content/zotero/components/utils.js"
source_code = source_by_file[file_path]
tree = ast_trees[file_path]

for node in walk_tree(tree.root_node):
    if node.type == "export_statement":
        if "export {" in node_text(node, source_code):
            print_ast(node, source_code, max_depth=5)

export_statement: 'export {\n\tnextHTMLID, noop, getDragTargetOrient, createDragHandler, scrollIntoViewIfNeeded, stopP...'
  export_clause: '{\n\tnextHTMLID, noop, getDragTargetOrient, createDragHandler, scrollIntoViewIfNeeded, stopPropagat...'
    export_specifier: 'nextHTMLID'
      identifier: 'nextHTMLID'
    export_specifier: 'noop'
      identifier: 'noop'
    export_specifier: 'getDragTargetOrient'
      identifier: 'getDragTargetOrient'
    export_specifier: 'createDragHandler'
      identifier: 'createDragHandler'
    export_specifier: 'scrollIntoViewIfNeeded'
      identifier: 'scrollIntoViewIfNeeded'
    export_specifier: 'stopPropagation'
      identifier: 'stopPropagation'
    export_specifier: 'pick'
      identifier: 'pick'
    export_specifier: 'omit'
      identifier: 'omit'


In [70]:
export_alias_patterns = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "export_specifier":
            continue

        identifiers = [
            child for child in node.named_children
            if child.type == "identifier"
        ]

        if len(identifiers) > 1:
            export_alias_patterns.append({
                "file_id": file_path,
                "specifier_text": node_text(node, source_code),
                "identifiers": [
                    node_text(child, source_code)
                    for child in identifiers
                ]
            })

export_alias_patterns_df = pd.DataFrame(export_alias_patterns)

print("Aliased export specifiers:", len(export_alias_patterns_df))
display(export_alias_patterns_df)

Aliased export specifiers: 0


""


In [71]:
for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "export_statement":
            continue

        for child in node.named_children:
            if child.type in ("lexical_declaration", "variable_declaration"):
                print(f"\nFILE: {file_path}")
                print_ast(node, source_code, max_depth=5)


FILE: chrome/content/zotero/actors/TranslationParent.mjs
export_statement: 'export const TranslationManager = new (class {\n\t_registeredRemoteTranslates = new Map();\n\t\n\tadd(i...'
  lexical_declaration: 'const TranslationManager = new (class {\n\t_registeredRemoteTranslates = new Map();\n\t\n\tadd(id, remo...'
    variable_declarator: 'TranslationManager = new (class {\n\t_registeredRemoteTranslates = new Map();\n\t\n\tadd(id, remoteTran...'
      identifier: 'TranslationManager'
      new_expression: 'new (class {\n\t_registeredRemoteTranslates = new Map();\n\t\n\tadd(id, remoteTranslate) {\n\t\tthis._regi...'
        parenthesized_expression: '(class {\n\t_registeredRemoteTranslates = new Map();\n\t\n\tadd(id, remoteTranslate) {\n\t\tthis._register...'
          class: 'class {\n\t_registeredRemoteTranslates = new Map();\n\t\n\tadd(id, remoteTranslate) {\n\t\tthis._registere...'

FILE: chrome/content/zotero/import/mendeley/mendeleyImport.mjs
export_statement: 'export var Zotero_

In [73]:
declaration_export_patterns = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "export_statement":
            continue

        for declaration in node.named_children:
            if declaration.type not in (
                "lexical_declaration",
                "variable_declaration"
            ):
                continue

            declarators = [
                child for child in declaration.named_children
                if child.type == "variable_declarator"
            ]

            declaration_export_patterns.append({
                "file_id": file_path,
                "declaration_type": declaration.type,
                "declarator_count": len(declarators),
                "declarator_types": [
                    child.named_children[0].type
                    if child.named_children else None
                    for child in declarators
                ],
                "export_text": node_text(node, source_code)
            })

declaration_export_patterns_df = pd.DataFrame(
    declaration_export_patterns
)

print("Exported variable declarations:", len(declaration_export_patterns_df))

print("\nDeclarator counts:")
display(
    declaration_export_patterns_df["declarator_count"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

print("\nDeclarator pattern types:")
display(
    declaration_export_patterns_df.explode("declarator_types")[
        "declarator_types"
    ].value_counts().to_frame("count")
)

print("\nDeclarations with multiple declarators or non-identifier patterns:")
display(
    declaration_export_patterns_df[
        (declaration_export_patterns_df["declarator_count"] > 1)
        | declaration_export_patterns_df["declarator_types"].apply(
            lambda types: any(t != "identifier" for t in types)
        )
    ][["file_id", "declarator_count", "declarator_types", "export_text"]]
)

Exported variable declarations: 17

Declarator counts:


,count
declarator_count,
1,17



Declarator pattern types:


,count
declarator_types,
identifier,17



Declarations with multiple declarators or non-identifier patterns:


,file_id,declarator_count,declarator_types,export_text


In [74]:
default_export_patterns = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "export_statement":
            continue

        export_text = node_text(node, source_code)

        if export_text.startswith("export default"):
            default_export_patterns.append({
                "file_id": file_path,
                "child_types": [
                    child.type for child in node.named_children
                ],
                "export_text": export_text
            })

default_exports_df = pd.DataFrame(default_export_patterns)

print("Default exports:", len(default_exports_df))

display(
    default_exports_df[
        ["file_id", "child_types", "export_text"]
    ].to_string(index=False)
)

Default exports: 9


'                                                file_id       child_types                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [75]:
def extract_es_exports_with_bindings(file_id, tree, source_code):

    export_records = []
    binding_records = []

    for node in walk_tree(tree.root_node):
        if node.type != "export_statement":
            continue

        export_text = node_text(node, source_code)
        export_id = stable_id(file_id, node.start_byte, "es_export", prefix="exp_")

        named_children = node.named_children

        # Identify default export using the source prefix.
        is_default = export_text.startswith("export default")

        # Detect a source module for re-exports.
        source_node = next(
            (child for child in named_children if child.type == "string"),
            None
        )

        module_specifier = (
            node_text(source_node, source_code).strip("'\"")
            if source_node else None
        )

        export_records.append({
            "export_id": export_id,
            "file_id": file_id,
            "export_kind": "default" if is_default else "named",
            "export_text": export_text,
            "module_specifier": module_specifier,
            "start_line": node.start_point.row + 1,
            "start_column": node.start_point.column + 1,
            "end_line": node.end_point.row + 1,
            "end_column": node.end_point.column + 1,
            "start_byte": node.start_byte,
            "end_byte": node.end_byte,
        })

        # Default exports
        if is_default:
            expression = named_children[0] if named_children else None

            local_name = (
                node_text(expression, source_code)
                if expression and expression.type == "identifier"
                else None
            )

            binding_records.append({
                "export_binding_id": stable_id(
                    export_id, "default", prefix="exb_"
                ),
                "export_id": export_id,
                "local_name": local_name,
                "exported_name": "default",
                "binding_kind": (
                    "identifier"
                    if local_name else "expression"
                ),
                "source_text": (
                    node_text(expression, source_code)
                    if expression else None
                ),
            })

            continue

        # Declaration exports: export const/function/class ...
        declaration = next(
            (
                child for child in named_children
                if child.type in (
                    "lexical_declaration",
                    "variable_declaration",
                    "function_declaration",
                    "class_declaration",
                )
            ),
            None
        )

        if declaration:
            if declaration.type in (
                "lexical_declaration",
                "variable_declaration",
            ):
                for declarator in declaration.named_children:
                    if declarator.type != "variable_declarator":
                        continue

                    name_node = next(
                        (
                            child for child in declarator.named_children
                            if child.type == "identifier"
                        ),
                        None
                    )

                    local_name = (
                        node_text(name_node, source_code)
                        if name_node else None
                    )

                    if local_name:
                        binding_records.append({
                            "export_binding_id": stable_id(
                                export_id, local_name, prefix="exb_"
                            ),
                            "export_id": export_id,
                            "local_name": local_name,
                            "exported_name": local_name,
                            "binding_kind": "declaration",
                            "source_text": node_text(
                                declarator, source_code
                            ),
                        })

            else:
                name_node = next(
                    (
                        child for child in declaration.named_children
                        if child.type == "identifier"
                    ),
                    None
                )

                local_name = (
                    node_text(name_node, source_code)
                    if name_node else None
                )

                if local_name:
                    binding_records.append({
                        "export_binding_id": stable_id(
                            export_id, local_name, prefix="exb_"
                        ),
                        "export_id": export_id,
                        "local_name": local_name,
                        "exported_name": local_name,
                        "binding_kind": "declaration",
                        "source_text": node_text(
                            declaration, source_code
                        ),
                    })

            continue

        # Export clauses: export { foo, bar as baz }
        export_clause = next(
            (
                child for child in named_children
                if child.type == "export_clause"
            ),
            None
        )

        if export_clause:
            for specifier in export_clause.named_children:
                if specifier.type != "export_specifier":
                    continue

                identifiers = [
                    child for child in specifier.named_children
                    if child.type == "identifier"
                ]

                if not identifiers:
                    continue

                local_name = node_text(identifiers[0], source_code)
                exported_name = (
                    node_text(identifiers[-1], source_code)
                )

                binding_records.append({
                    "export_binding_id": stable_id(
                        export_id, local_name, exported_name, prefix="exb_"
                    ),
                    "export_id": export_id,
                    "local_name": local_name,
                    "exported_name": exported_name,
                    "binding_kind": (
                        "re_export"
                        if module_specifier else "export_clause"
                    ),
                    "source_text": node_text(specifier, source_code),
                })

    return export_records, binding_records


# Extract across the repository
es_export_records = []
es_export_binding_records = []

for file_path, tree in ast_trees.items():

    exports, bindings = extract_es_exports_with_bindings(
        file_path,
        tree,
        source_by_file[file_path]
    )

    es_export_records.extend(exports)
    es_export_binding_records.extend(bindings)


es_exports_df = pd.DataFrame(es_export_records)
es_export_bindings_df = pd.DataFrame(es_export_binding_records)

print("ES export statements:", len(es_exports_df))
print("ES export bindings:", len(es_export_bindings_df))

print("\nExport statement types:")
display(es_exports_df["export_kind"].value_counts().to_frame("count"))

print("\nBinding types:")
display(es_export_bindings_df["binding_kind"].value_counts().to_frame("count"))

ES export statements: 96
ES export bindings: 104

Export statement types:


,count
export_kind,
named,87
default,9



Binding types:


,count
binding_kind,
declaration,82
export_clause,13
expression,6
identifier,3


In [76]:
print("EXPORT VALIDATION")
print("=" * 60)

print("Export statements:", len(es_exports_df))
print("Export bindings:", len(es_export_bindings_df))

print(
    "Duplicate export IDs:",
    es_exports_df["export_id"].duplicated().sum()
)

print(
    "Duplicate binding IDs:",
    es_export_bindings_df["export_binding_id"].duplicated().sum()
)

print(
    "Exports without bindings:",
    len(
        set(es_exports_df["export_id"])
        - set(es_export_bindings_df["export_id"])
    )
)

print("\nBindings by type:")
display(
    es_export_bindings_df["binding_kind"]
    .value_counts()
    .to_frame("count")
)

print("\nBindings by export kind:")
display(
    es_exports_df.merge(
        es_export_bindings_df,
        on="export_id"
    )["export_kind"].value_counts().to_frame("count")
)

print("\nDefault exports:")
display(
    es_export_bindings_df[
        es_export_bindings_df["exported_name"] == "default"
    ][["local_name", "binding_kind", "source_text"]]
)

EXPORT VALIDATION
Export statements: 96
Export bindings: 104
Duplicate export IDs: 0
Duplicate binding IDs: 0
Exports without bindings: 0

Bindings by type:


,count
binding_kind,
declaration,82
export_clause,13
expression,6
identifier,3



Bindings by export kind:


,count
export_kind,
named,95
default,9



Default exports:


,local_name,binding_kind,source_text
25,NaN,expression,"React.forwardRef((props, ref) => <Editable\n\t..."
26,EditableContent,identifier,EditableContent
27,NaN,expression,"React.forwardRef((props, ref) => <Input\n\tinn..."
28,SelectInput,identifier,SelectInput
29,NaN,expression,"React.forwardRef((props, ref) => <TextAreaInpu..."
30,NaN,expression,memo(ProgressQueueTable)
31,TabBar,identifier,TabBar
90,NaN,expression,"[\n\t...compat.extends(""@zotero""),\n\t\n\t{\n\..."
92,NaN,expression,require_index()


In [77]:
display(
    es_export_bindings_df["binding_kind"]
    .value_counts()
    .to_frame("count")
)

,count
binding_kind,
declaration,82
export_clause,13
expression,6
identifier,3


In [78]:
commonjs_export_patterns = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "assignment_expression":
            continue

        if not node.named_children:
            continue

        left = node.named_children[0]

        if left.type != "member_expression":
            continue

        left_text = node_text(left, source_code).replace(" ", "")

        if (
            left_text.startswith("module.exports")
            or left_text.startswith("exports.")
        ):
            commonjs_export_patterns.append({
                "file_id": file_path,
                "left_expression": left_text,
                "export_text": node_text(node, source_code),
            })

commonjs_export_patterns_df = pd.DataFrame(
    commonjs_export_patterns
)

print("CommonJS export assignments:", len(commonjs_export_patterns_df))

display(
    commonjs_export_patterns_df[
        ["file_id", "left_expression", "export_text"]
    ].head(30)
)

print("\nAssignment targets:")
display(
    commonjs_export_patterns_df["left_expression"]
    .value_counts()
    .head(30)
    .to_frame("count")
)

CommonJS export assignments: 130


,file_id,left_expression,export_text
0,chrome/content/zotero/collectionTree.jsx,module.exports,module.exports = CollectionTree
1,chrome/content/zotero/collectionViewItemTree.jsx,module.exports,module.exports = CollectionViewItemTree
2,chrome/content/zotero/components/button.jsx,module.exports,"module.exports = {\n\tButtonGroup,\n\tButton\n}"
3,chrome/content/zotero/components/draggable.jsx,module.exports,module.exports = Draggable
4,chrome/content/zotero/components/icons.jsx,module.exports,"module.exports = { Icon, CSSIcon, CSSItemTypeI..."
5,chrome/content/zotero/components/icons.jsx,module.exports.getCSSIcon,module.exports.getCSSIcon = function (key) {\n...
6,chrome/content/zotero/components/icons.jsx,module.exports.getCSSItemTypeIcon,module.exports.getCSSItemTypeIcon = function (...
7,chrome/content/zotero/components/search.jsx,module.exports,module.exports = Search
8,chrome/content/zotero/components/tagSelector.jsx,module.exports,module.exports = TagSelector
9,chrome/content/zotero/components/tagSelector/t...,module.exports,module.exports = TagList



Assignment targets:


,count
left_expression,
module.exports,46
module.exports.doTranslate,3
module.exports.ItemTreeRow,2
exports.BURMESE_UNICODE_RANGE,2
exports.CJK_UNICODE_RANGES,2
exports.JAVANESE_UNICODE_RANGE,2
exports.KHMER_UNICODE_RANGE,2
exports.LAO_UNICODE_RANGE,2
exports.THAI_UNICODE_RANGE,2


In [81]:
def classify_commonjs_export_target(target):
    if target == "module.exports":
        return "module_assignment"
    elif target.startswith("module.exports."):
        return "module_property_assignment"
    elif target.startswith("exports."):
        return "exports_property_assignment"
    return "other"


commonjs_export_patterns_df["assignment_type"] = (
    commonjs_export_patterns_df["left_expression"]
    .apply(classify_commonjs_export_target)
)

print("Assignment types:")
display(
    commonjs_export_patterns_df["assignment_type"]
    .value_counts()
    .to_frame("count")
)

print("\nSample assignments:")
display(
    commonjs_export_patterns_df[
        ["file_id", "left_expression", "export_text", "assignment_type"]
    ].head(20)
)

Assignment types:


,count
assignment_type,
exports_property_assignment,62
module_assignment,46
module_property_assignment,22



Sample assignments:


,file_id,left_expression,export_text,assignment_type
0,chrome/content/zotero/collectionTree.jsx,module.exports,module.exports = CollectionTree,module_assignment
1,chrome/content/zotero/collectionViewItemTree.jsx,module.exports,module.exports = CollectionViewItemTree,module_assignment
2,chrome/content/zotero/components/button.jsx,module.exports,"module.exports = {\n\tButtonGroup,\n\tButton\n}",module_assignment
3,chrome/content/zotero/components/draggable.jsx,module.exports,module.exports = Draggable,module_assignment
4,chrome/content/zotero/components/icons.jsx,module.exports,"module.exports = { Icon, CSSIcon, CSSItemTypeI...",module_assignment
5,chrome/content/zotero/components/icons.jsx,module.exports.getCSSIcon,module.exports.getCSSIcon = function (key) {\n...,module_property_assignment
6,chrome/content/zotero/components/icons.jsx,module.exports.getCSSItemTypeIcon,module.exports.getCSSItemTypeIcon = function (...,module_property_assignment
7,chrome/content/zotero/components/search.jsx,module.exports,module.exports = Search,module_assignment
8,chrome/content/zotero/components/tagSelector.jsx,module.exports,module.exports = TagSelector,module_assignment
9,chrome/content/zotero/components/tagSelector/t...,module.exports,module.exports = TagList,module_assignment


In [82]:
commonjs_export_ast_samples = []

for file_path, tree in ast_trees.items():
    source_code = source_by_file[file_path]

    for node in walk_tree(tree.root_node):
        if node.type != "assignment_expression":
            continue

        if not node.named_children:
            continue

        left = node.named_children[0]

        if left.type != "member_expression":
            continue

        left_text = node_text(left, source_code).replace(" ", "")

        if not (
            left_text == "module.exports"
            or left_text.startswith("module.exports.")
            or left_text.startswith("exports.")
        ):
            continue

        right = (
            node.named_children[1]
            if len(node.named_children) > 1
            else None
        )

        commonjs_export_ast_samples.append({
            "file_path": file_path,
            "assignment": node_text(node, source_code),
            "left_type": left.type,
            "left_text": node_text(left, source_code),
            "left_children": [
                (child.type, node_text(child, source_code))
                for child in left.named_children
            ],
            "right_type": right.type if right else None,
            "right_text": node_text(right, source_code) if right else None,
        })

commonjs_export_ast_samples_df = pd.DataFrame(
    commonjs_export_ast_samples
)

display(
    commonjs_export_ast_samples_df[
        [
            "assignment",
            "left_children",
            "right_type",
            "right_text",
        ]
    ].head(15)
)

,assignment,left_children,right_type,right_text
0,module.exports = CollectionTree,"[(identifier, module), (property_identifier, e...",identifier,CollectionTree
1,module.exports = CollectionViewItemTree,"[(identifier, module), (property_identifier, e...",identifier,CollectionViewItemTree
2,"module.exports = {\n\tButtonGroup,\n\tButton\n}","[(identifier, module), (property_identifier, e...",object,"{\n\tButtonGroup,\n\tButton\n}"
3,module.exports = Draggable,"[(identifier, module), (property_identifier, e...",identifier,Draggable
4,"module.exports = { Icon, CSSIcon, CSSItemTypeI...","[(identifier, module), (property_identifier, e...",object,"{ Icon, CSSIcon, CSSItemTypeIcon }"
5,module.exports.getCSSIcon = function (key) {\n...,"[(member_expression, module.exports), (propert...",function_expression,function (key) {\n\tif (!cssIconsCache.has(key...
6,module.exports.getCSSItemTypeIcon = function (...,"[(member_expression, module.exports), (propert...",function_expression,"function (itemType, key = 'item-type') {\n\tle..."
7,module.exports = Search,"[(identifier, module), (property_identifier, e...",identifier,Search
8,module.exports = TagSelector,"[(identifier, module), (property_identifier, e...",identifier,TagSelector
9,module.exports = TagList,"[(identifier, module), (property_identifier, e...",identifier,TagList


In [83]:
def extract_commonjs_exports_with_bindings(file_id, tree, source_code):

    export_records = []
    binding_records = []

    for node in walk_tree(tree.root_node):

        if node.type != "assignment_expression":
            continue

        named_children = node.named_children

        if len(named_children) < 2:
            continue

        left = named_children[0]
        right = named_children[1]

        if left.type != "member_expression":
            continue

        left_text = node_text(left, source_code).replace(" ", "")

        # Identify the CommonJS export target.
        if left_text == "module.exports":
            export_target = "module.exports"
            assignment_type = "module_assignment"
            exported_name = "default"

        elif left_text.startswith("module.exports."):
            export_target = "module.exports"
            assignment_type = "module_property_assignment"
            exported_name = left_text[len("module.exports."):]

        elif left_text.startswith("exports."):
            export_target = "exports"
            assignment_type = "exports_property_assignment"
            exported_name = left_text[len("exports."):]

        else:
            continue

        export_text = node_text(node, source_code)

        export_id = stable_id(
            file_id,
            node.start_byte,
            "commonjs_export",
            prefix="exp_"
        )

        binding_id = stable_id(
            export_id,
            exported_name,
            prefix="exb_"
        )

        export_records.append({
            "export_id": export_id,
            "file_id": file_id,
            "export_system": "commonjs",
            "assignment_type": assignment_type,
            "export_target": export_target,
            "export_text": export_text,
            "start_line": node.start_point.row + 1,
            "start_column": node.start_point.column + 1,
            "end_line": node.end_point.row + 1,
            "end_column": node.end_point.column + 1,
            "start_byte": node.start_byte,
            "end_byte": node.end_byte,
        })

        binding_records.append({
            "export_binding_id": binding_id,
            "export_id": export_id,
            "local_name": (
                node_text(right, source_code)
                if right.type == "identifier"
                else None
            ),
            "exported_name": exported_name,
            "binding_kind": (
                "identifier"
                if right.type == "identifier"
                else "expression"
            ),
            "expression_type": right.type,
            "source_text": node_text(right, source_code),
        })

    return export_records, binding_records

In [84]:
commonjs_export_records = []
commonjs_export_binding_records = []

for file_path, tree in ast_trees.items():

    exports, bindings = extract_commonjs_exports_with_bindings(
        file_path,
        tree,
        source_by_file[file_path]
    )

    commonjs_export_records.extend(exports)
    commonjs_export_binding_records.extend(bindings)


commonjs_exports_df = pd.DataFrame(commonjs_export_records)
commonjs_export_bindings_df = pd.DataFrame(
    commonjs_export_binding_records
)

print("CommonJS export statements:", len(commonjs_exports_df))
print("CommonJS export bindings:", len(commonjs_export_bindings_df))

print("\nAssignment types:")
display(
    commonjs_exports_df["assignment_type"]
    .value_counts()
    .to_frame("count")
)

print("\nBinding types:")
display(
    commonjs_export_bindings_df["binding_kind"]
    .value_counts()
    .to_frame("count")
)

CommonJS export statements: 130
CommonJS export bindings: 130

Assignment types:


,count
assignment_type,
exports_property_assignment,62
module_assignment,46
module_property_assignment,22



Binding types:


,count
binding_kind,
identifier,83
expression,47


In [85]:
print("COMMONJS EXPORT VALIDATION")
print("=" * 60)

print("Export statements:", len(commonjs_exports_df))
print("Export bindings:", len(commonjs_export_bindings_df))

print(
    "Duplicate export IDs:",
    commonjs_exports_df["export_id"].duplicated().sum()
)

print(
    "Duplicate binding IDs:",
    commonjs_export_bindings_df["export_binding_id"].duplicated().sum()
)

print(
    "Exports without bindings:",
    len(
        set(commonjs_exports_df["export_id"])
        - set(commonjs_export_bindings_df["export_id"])
    )
)

print("\nAssignment types:")
display(
    commonjs_exports_df["assignment_type"]
    .value_counts()
    .to_frame("count")
)

print("\nBinding types:")
display(
    commonjs_export_bindings_df["binding_kind"]
    .value_counts()
    .to_frame("count")
)

print("\nExpression types:")
display(
    commonjs_export_bindings_df["expression_type"]
    .value_counts()
    .to_frame("count")
)

COMMONJS EXPORT VALIDATION
Export statements: 130
Export bindings: 130
Duplicate export IDs: 0
Duplicate binding IDs: 0
Exports without bindings: 0

Assignment types:


,count
assignment_type,
exports_property_assignment,62
module_assignment,46
module_property_assignment,22



Binding types:


,count
binding_kind,
identifier,83
expression,47



Expression types:


,count
expression_type,
identifier,83
function_expression,15
unary_expression,9
array,8
object,7
member_expression,3
class,1
assignment_expression,1
binary_expression,1


In [86]:
# Combine ES module and CommonJS export statements
exports_all_df = pd.concat(
    [es_exports_df, commonjs_exports_df],
    ignore_index=True,
    sort=False
)

# Combine ES module and CommonJS export bindings
export_bindings_all_df = pd.concat(
    [es_export_bindings_df, commonjs_export_bindings_df],
    ignore_index=True,
    sort=False
)

print("COMBINED EXPORT SUMMARY")
print("=" * 60)

print("Export statements:", len(exports_all_df))
print("Export bindings:", len(export_bindings_all_df))

print("\nExport systems:")
display(
    exports_all_df["export_system"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nExport assignment types:")
display(
    exports_all_df["assignment_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nBinding types:")
display(
    export_bindings_all_df["binding_kind"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nDuplicate IDs:")
print(
    "Export IDs:",
    exports_all_df["export_id"].duplicated().sum()
)
print(
    "Binding IDs:",
    export_bindings_all_df["export_binding_id"].duplicated().sum()
)

COMBINED EXPORT SUMMARY
Export statements: 226
Export bindings: 234

Export systems:


,count
export_system,
commonjs,130
NaN,96



Export assignment types:


,count
assignment_type,
NaN,96
exports_property_assignment,62
module_assignment,46
module_property_assignment,22



Binding types:


,count
binding_kind,
identifier,86
declaration,82
expression,53
export_clause,13



Duplicate IDs:
Export IDs: 0
Binding IDs: 0


In [89]:
es_exports_df["export_system"] = "es_module"

exports_all_df = pd.concat(
    [es_exports_df, commonjs_exports_df],
    ignore_index=True,
    sort=False
)

In [90]:
display(
    exports_all_df["export_system"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
export_system,
commonjs,130
es_module,96


In [91]:
display(
    symbols_df[
        symbols_df["name"].isin([
            "CollectionTree",
            "renderCell",
            "VirtualizedTree",
            "Button",
            "getCSSIcon",
        ])
    ][
        [
            "symbol_id",
            "file_id",
            "name",
            "qualified_name",
            "symbol_type",
            "declaration_type",
            "scope_id",
            "start_line",
            "source_text",
        ]
    ].head(30)
)

,symbol_id,file_id,name,qualified_name,symbol_type,declaration_type,scope_id,start_line,source_text
1246,sym_4bf4f7e257edd54b,chrome/content/zotero/collectionTree.jsx,VirtualizedTree,None,variable,lexical_declaration,scope_6979f95b75d4c23c,29,VirtualizedTree = require('components/virtuali...
1252,sym_f995df72fd05a10b,chrome/content/zotero/collectionTree.jsx,CollectionTree,None,variable,variable_declaration,scope_6979f95b75d4c23c,37,CollectionTree = class CollectionTree extends ...
1253,sym_7a82feced941ae2d,chrome/content/zotero/collectionTree.jsx,CollectionTree,None,class,class,scope_c18304e686f5c0a4,37,class CollectionTree extends LibraryTree {\n\t...
2132,sym_790700c9564efc37,chrome/content/zotero/components/button.jsx,Button,None,class,class_declaration,scope_4cf49e6e4394009a,45,class Button extends PureComponent {\n\tget cl...
2827,sym_5ba4c429965b1b7c,chrome/content/zotero/components/virtualized-t...,VirtualizedTree,None,class,class_declaration,scope_62975ba95bce3d98,1749,class VirtualizedTree extends VirtualizedTable...
2919,sym_65eeca378ceb7d6e,chrome/content/zotero/components/virtualized-t...,renderCell,None,function,function_declaration,scope_99dee5252c126031,2186,"function renderCell(index, data, column, dir =..."
7941,sym_3bc2fb50c0598289,chrome/content/zotero/integration/citationDial...,CollectionTree,None,variable,lexical_declaration,scope_d55fbd6bf6c6a7af,787,CollectionTree = require('zotero/collectionTree')
9455,sym_e28bba6f8803e95f,chrome/content/zotero/itemTreeRow.js,renderCell,None,method,method_definition,scope_ee292162fe161c5f,113,"renderCell(index, data, column, isFirstColumn)..."
9479,sym_04e58e063a1799e5,chrome/content/zotero/itemTreeRow.js,renderCell,None,method,method_definition,scope_17b410d73fa3b5c4,271,"renderCell(index, data, column, isFirstColumn,..."
30620,sym_b6a3e15b3c4fa392,chrome/content/zotero/zoteroPane.js,CollectionTree,None,variable,lexical_declaration,scope_92dd2690cc587103,1656,CollectionTree = require('zotero/collectionTree')


In [92]:
# Attach file information to each export binding
export_binding_candidates_df = export_bindings_all_df.merge(
    exports_all_df[
        ["export_id", "file_id", "export_system", "export_text"]
    ],
    on="export_id",
    how="left"
)

# Keep bindings that identify a local symbol by name
named_bindings_df = export_binding_candidates_df[
    export_binding_candidates_df["local_name"].notna()
].copy()

# Match against symbols in the same file
export_symbol_candidates_df = named_bindings_df.merge(
    symbols_df,
    left_on=["file_id", "local_name"],
    right_on=["file_id", "name"],
    how="left",
    suffixes=("_export", "_symbol")
)

# Summarise the number of symbol candidates per export binding
export_symbol_match_counts_df = (
    export_symbol_candidates_df
    .groupby("export_binding_id")
    .agg(
        candidate_count=("symbol_id", "count"),
        local_name=("local_name", "first"),
        file_id=("file_id", "first"),
        binding_kind=("binding_kind", "first"),
        export_system=("export_system", "first"),
    )
    .reset_index()
)

print("EXPORT-TO-SYMBOL MATCH SUMMARY")
print("=" * 60)

display(
    export_symbol_match_counts_df["candidate_count"]
    .value_counts()
    .sort_index()
    .to_frame("binding_count")
)

print("\nTotal named export bindings:", len(named_bindings_df))
print(
    "Bindings with at least one candidate:",
    (export_symbol_match_counts_df["candidate_count"] > 0).sum()
)
print(
    "Bindings with multiple candidates:",
    (export_symbol_match_counts_df["candidate_count"] > 1).sum()
)
print(
    "Bindings without candidates:",
    (export_symbol_match_counts_df["candidate_count"] == 0).sum()
)

EXPORT-TO-SYMBOL MATCH SUMMARY


,binding_count
candidate_count,
0,7
1,162
2,11
150,1



Total named export bindings: 181
Bindings with at least one candidate: 174
Bindings with multiple candidates: 12
Bindings without candidates: 7


In [96]:
# Identify export bindings with multiple symbol candidates
ambiguous_binding_ids = (
    export_symbol_match_counts_df.loc[
        export_symbol_match_counts_df["candidate_count"] > 1,
        "export_binding_id"
    ]
)

ambiguous_exports_df = (
    export_symbol_candidates_df[
        export_symbol_candidates_df["export_binding_id"].isin(
            ambiguous_binding_ids
        )
    ]
    .sort_values(
        ["file_id", "local_name", "start_line"]
    )
)

display(
    ambiguous_exports_df[
        [
            "file_id",
            "local_name",
            "symbol_id",
            "symbol_type",
            "declaration_type",
            "scope_id",
            "start_line",
            "end_line",
            "source_text_symbol",
        ]
    ]
)

,file_id,local_name,symbol_id,symbol_type,declaration_type,scope_id,start_line,end_line,source_text_symbol
98,chrome/content/zotero/collectionTree.jsx,CollectionTree,sym_f995df72fd05a10b,variable,variable_declaration,scope_6979f95b75d4c23c,37.0,3532.0,CollectionTree = class CollectionTree extends ...
99,chrome/content/zotero/collectionTree.jsx,CollectionTree,sym_7a82feced941ae2d,class,class,scope_c18304e686f5c0a4,37.0,3532.0,class CollectionTree extends LibraryTree {\n\t...
113,chrome/content/zotero/itemTree.jsx,ItemTree,sym_b9fd5b222944abc7,variable,variable_declaration,scope_962b8181cafa1410,894.0,2946.0,ItemTree = class ItemTree extends LibraryTree ...
114,chrome/content/zotero/itemTree.jsx,ItemTree,sym_1c50637eb5a0c679,class,class,scope_6aa7e53c1c473b4b,894.0,2946.0,class ItemTree extends LibraryTree {\n\tstatic...
126,chrome/content/zotero/libraryTree.js,LibraryTree,sym_52b353f73d1f1499,variable,variable_declaration,scope_01a84dd45f29fbec,33.0,305.0,LibraryTree = class LibraryTree extends React....
...,...,...,...,...,...,...,...,...,...
336,test/resource/pako_inflate.js,inflate,sym_d0b149cd27d8a9eb,function,function_declaration,scope_5789d762d0ac570c,1154.0,2244.0,"function inflate(strm, flush) {\n var state;\..."
339,test/resource/pako_inflate.js,inflate,sym_d0b149cd27d8a9eb,function,function_declaration,scope_5789d762d0ac570c,1154.0,2244.0,"function inflate(strm, flush) {\n var state;\..."
330,test/resource/pako_inflate.js,inflate,sym_fb385ee90c92d2c8,function,function_declaration,scope_8258d136e61920be,3032.0,3041.0,"function inflate(input, options) {\n var infl..."
337,test/resource/pako_inflate.js,inflate,sym_fb385ee90c92d2c8,function,function_declaration,scope_8258d136e61920be,3032.0,3041.0,"function inflate(input, options) {\n var infl..."


In [97]:
ambiguous_diagnostic_df = (
    export_symbol_candidates_df[
        export_symbol_candidates_df["export_binding_id"].isin(
            ambiguous_binding_ids
        )
    ]
    .groupby(
        [
            "export_binding_id",
            "file_id",
            "local_name",
            "binding_kind",
            "export_system",
        ],
        dropna=False
    )
    .agg(
        candidate_rows=("symbol_id", "size"),
        unique_symbols=("symbol_id", "nunique"),
        symbol_ids=("symbol_id", lambda x: list(x.dropna().unique())),
        symbol_types=("symbol_type", lambda x: list(x.dropna().unique())),
        declaration_types=(
            "declaration_type",
            lambda x: list(x.dropna().unique())
        ),
    )
    .reset_index()
    .sort_values(["file_id", "local_name"])
)

display(ambiguous_diagnostic_df)

,export_binding_id,file_id,local_name,binding_kind,export_system,candidate_rows,unique_symbols,symbol_ids,symbol_types,declaration_types
7,exb_a7db9fb3236ba891,chrome/content/zotero/collectionTree.jsx,CollectionTree,identifier,commonjs,2,2,"[sym_f995df72fd05a10b, sym_7a82feced941ae2d]","[variable, class]","[variable_declaration, class]"
1,exb_29e811501662f5a2,chrome/content/zotero/itemTree.jsx,ItemTree,identifier,commonjs,2,2,"[sym_b9fd5b222944abc7, sym_1c50637eb5a0c679]","[variable, class]","[variable_declaration, class]"
4,exb_605216542a295143,chrome/content/zotero/libraryTree.js,LibraryTree,identifier,commonjs,2,2,"[sym_52b353f73d1f1499, sym_59bc1777fc09b1f1]","[variable, class]","[variable_declaration, class]"
10,exb_c4afd3f9ed00e2dd,resource/loader.sys.mjs,data,identifier,commonjs,2,2,"[sym_dc22f24ff5de4a2e, sym_5e0d17e41d8f3ddb]",[variable],[lexical_declaration]
11,exb_f9b8a6af6afcba74,resource/tinymce/tinymce.min.js,e,identifier,commonjs,150,150,"[sym_5165c8ba0863fb07, sym_189b65c834829404, s...","[variable, function_expression]","[variable_declaration, function_expression]"
2,exb_2d345ae5a3eb1827,test/resource/pako_inflate.js,GZheader,identifier,commonjs,2,2,"[sym_e7d49d5947899024, sym_3d5579ae5197b27b]","[function, variable]","[function_declaration, variable_declaration]"
9,exb_bdfd970dd9cbe6c3,test/resource/pako_inflate.js,ZStream,identifier,commonjs,2,2,"[sym_cfd0ee028be241f9, sym_c25825bcec9f85cc]","[function, variable]","[function_declaration, variable_declaration]"
6,exb_7107b1c0090b3a8c,test/resource/pako_inflate.js,adler32,identifier,commonjs,2,2,"[sym_eb984baab62b81c3, sym_a62970a8ddb30ad7]","[function, variable]","[function_declaration, variable_declaration]"
5,exb_667da70c81909d62,test/resource/pako_inflate.js,crc32,identifier,commonjs,2,2,"[sym_c4ced127a512de24, sym_96083a31922ea627]","[function, variable]","[function_declaration, variable_declaration]"
0,exb_006683bac1b974f6,test/resource/pako_inflate.js,inflate,identifier,commonjs,2,2,"[sym_d0b149cd27d8a9eb, sym_fb385ee90c92d2c8]",[function],[function_declaration]


In [98]:
ambiguous_detail_df = (
    export_symbol_candidates_df[
        export_symbol_candidates_df["export_binding_id"].isin(
            ambiguous_binding_ids
        )
    ]
    .drop_duplicates(
        ["export_binding_id", "symbol_id"]
    )
    [
        [
            "export_binding_id",
            "file_id",
            "local_name",
            "symbol_id",
            "symbol_type",
            "declaration_type",
            "scope_id",
            "start_line",
            "end_line",
            "source_text_symbol",
        ]
    ]
    .sort_values(
        ["file_id", "local_name", "start_line"]
    )
)

display(ambiguous_detail_df.to_string(index=False))

'   export_binding_id                                  file_id     local_name            symbol_id         symbol_type     declaration_type               scope_id  start_line  end_line                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [99]:
# One row per export binding and symbol candidate
ambiguous_summary_df = (
    export_symbol_candidates_df[
        export_symbol_candidates_df["export_binding_id"].isin(
            ambiguous_binding_ids
        )
    ]
    .groupby(
        ["file_id", "local_name", "export_binding_id"],
        dropna=False
    )
    .agg(
        candidate_count=("symbol_id", "nunique"),
        candidate_types=(
            "symbol_type",
            lambda x: ", ".join(sorted(set(x.dropna())))
        ),
        declaration_types=(
            "declaration_type",
            lambda x: ", ".join(sorted(set(x.dropna())))
        ),
    )
    .reset_index()
)

display(
    ambiguous_summary_df[
        [
            "file_id",
            "local_name",
            "candidate_count",
            "candidate_types",
            "declaration_types",
        ]
    ].to_string(index=False)
)

'                                 file_id     local_name  candidate_count               candidate_types                          declaration_types\nchrome/content/zotero/collectionTree.jsx CollectionTree                2               class, variable                class, variable_declaration\n      chrome/content/zotero/itemTree.jsx       ItemTree                2               class, variable                class, variable_declaration\n    chrome/content/zotero/libraryTree.js    LibraryTree                2               class, variable                class, variable_declaration\n                 resource/loader.sys.mjs           data                2                      variable                        lexical_declaration\n         resource/tinymce/tinymce.min.js              e              150 function_expression, variable  function_expression, variable_declaration\n           test/resource/pako_inflate.js       GZheader                2            function, variable function_decla

In [100]:
# Deduplicate candidate rows by export binding and symbol
unique_candidates_df = export_symbol_candidates_df.drop_duplicates(
    ["export_binding_id", "symbol_id"]
).copy()

# Summarise unique symbol candidates
resolution_diagnostic_df = (
    unique_candidates_df
    .groupby("export_binding_id")
    .agg(
        file_id=("file_id", "first"),
        local_name=("local_name", "first"),
        candidate_count=("symbol_id", "nunique"),
        candidate_types=(
            "symbol_type",
            lambda x: ", ".join(sorted(set(x.dropna())))
        ),
    )
    .reset_index()
)

display(
    resolution_diagnostic_df[
        resolution_diagnostic_df["candidate_count"] > 1
    ].sort_values(["file_id", "local_name"]).to_string(index=False)
)

'   export_binding_id                                  file_id     local_name  candidate_count               candidate_types\nexb_a7db9fb3236ba891 chrome/content/zotero/collectionTree.jsx CollectionTree                2               class, variable\nexb_29e811501662f5a2       chrome/content/zotero/itemTree.jsx       ItemTree                2               class, variable\nexb_605216542a295143     chrome/content/zotero/libraryTree.js    LibraryTree                2               class, variable\nexb_c4afd3f9ed00e2dd                  resource/loader.sys.mjs           data                2                      variable\nexb_f9b8a6af6afcba74          resource/tinymce/tinymce.min.js              e              150 function_expression, variable\nexb_2d345ae5a3eb1827            test/resource/pako_inflate.js       GZheader                2            function, variable\nexb_bdfd970dd9cbe6c3            test/resource/pako_inflate.js        ZStream                2            function, variable\

In [101]:
def resolve_export_candidates(group):

    candidates = group.drop_duplicates("symbol_id")

    if len(candidates) == 1:
        selected = candidates.iloc[0]
        return pd.Series({
            "resolved_symbol_id": selected["symbol_id"],
            "resolution_status": "resolved",
            "resolution_basis": "single_candidate",
        })

    types = set(candidates["symbol_type"].dropna())

    if types == {"class", "variable"}:
        selected = candidates[
            candidates["symbol_type"] == "variable"
        ].iloc[0]

        return pd.Series({
            "resolved_symbol_id": selected["symbol_id"],
            "resolution_status": "resolved",
            "resolution_basis": "variable_binding_over_class_expression",
        })

    if types == {"variable", "function_expression"}:
        selected = candidates[
            candidates["symbol_type"] == "variable"
        ].iloc[0]

        return pd.Series({
            "resolved_symbol_id": selected["symbol_id"],
            "resolution_status": "resolved",
            "resolution_basis": "variable_binding_over_function_expression",
        })

    return pd.Series({
        "resolved_symbol_id": None,
        "resolution_status": "ambiguous",
        "resolution_basis": "requires_declaration_analysis",
    })


export_resolution_df = (
    unique_candidates_df
    .groupby("export_binding_id", group_keys=False)
    .apply(resolve_export_candidates)
    .reset_index()
)

display(
    export_resolution_df["resolution_status"].value_counts()
)

resolution_status
resolved     173
ambiguous      8
Name: count, dtype: int64

In [102]:
unresolved_exports_df = (
    export_resolution_df[
        export_resolution_df["resolution_status"] != "resolved"
    ]
    .merge(
        export_binding_candidates_df[
            ["export_binding_id", "file_id", "local_name", "export_system"]
        ].drop_duplicates("export_binding_id"),
        on="export_binding_id",
        how="left",
    )
)

display(unresolved_exports_df.to_string(index=False))

'   export_binding_id resolved_symbol_id resolution_status              resolution_basis                       file_id local_name export_system\nexb_006683bac1b974f6                NaN         ambiguous requires_declaration_analysis test/resource/pako_inflate.js    inflate      commonjs\nexb_2d345ae5a3eb1827                NaN         ambiguous requires_declaration_analysis test/resource/pako_inflate.js   GZheader      commonjs\nexb_369c98c82f55805e                NaN         ambiguous requires_declaration_analysis test/resource/pako_inflate.js    inflate      commonjs\nexb_667da70c81909d62                NaN         ambiguous requires_declaration_analysis test/resource/pako_inflate.js      crc32      commonjs\nexb_7107b1c0090b3a8c                NaN         ambiguous requires_declaration_analysis test/resource/pako_inflate.js    adler32      commonjs\nexb_abbd8c54655bb4ba                NaN         ambiguous requires_declaration_analysis test/resource/pako_inflate.js    inflate      c

In [103]:
unresolved_candidate_details_df = (
    unique_candidates_df[
        unique_candidates_df["export_binding_id"].isin(
            unresolved_exports_df["export_binding_id"]
        )
    ]
    [
        [
            "export_binding_id",
            "file_id",
            "local_name",
            "symbol_id",
            "symbol_type",
            "declaration_type",
            "scope_id",
            "start_line",
            "end_line",
        ]
    ]
    .sort_values(["file_id", "local_name", "start_line"])
)

display(unresolved_candidate_details_df.to_string(index=False))

'   export_binding_id                       file_id local_name            symbol_id symbol_type     declaration_type               scope_id  start_line  end_line\nexb_c4afd3f9ed00e2dd       resource/loader.sys.mjs       data sym_dc22f24ff5de4a2e    variable  lexical_declaration scope_50b001a411b7bc42        81.0      83.0\nexb_c4afd3f9ed00e2dd       resource/loader.sys.mjs       data sym_5e0d17e41d8f3ddb    variable  lexical_declaration scope_5f40024f26517cae       364.0     364.0\nexb_2d345ae5a3eb1827 test/resource/pako_inflate.js   GZheader sym_e7d49d5947899024    function function_declaration scope_356dd0f61d8a59e1       425.0     459.0\nexb_2d345ae5a3eb1827 test/resource/pako_inflate.js   GZheader sym_3d5579ae5197b27b    variable variable_declaration scope_f726acd48853de3b      2681.0    2681.0\nexb_bdfd970dd9cbe6c3 test/resource/pako_inflate.js    ZStream sym_cfd0ee028be241f9    function function_declaration scope_eefa7d22b603477f      2644.0    2667.0\nexb_bdfd970dd9cbe6c3 test/r

In [104]:
# Add export assignment locations
export_locations_df = exports_all_df[
    ["export_id", "start_line", "end_line", "export_text"]
].rename(columns={
    "start_line": "export_start_line",
    "end_line": "export_end_line",
})

# Attach export locations to the unresolved bindings
unresolved_detail_df = (
    unresolved_candidate_details_df
    .merge(
        export_binding_candidates_df[
            ["export_binding_id", "export_id"]
        ].drop_duplicates(),
        on="export_binding_id",
        how="left",
    )
    .merge(
        export_locations_df,
        on="export_id",
        how="left",
    )
)

# Attach lexical scope metadata
unresolved_detail_df = unresolved_detail_df.merge(
    scopes_df[
        ["scope_id", "scope_type", "parent_scope_id"]
    ],
    on="scope_id",
    how="left",
)

display(
    unresolved_detail_df[
        [
            "file_id",
            "local_name",
            "export_start_line",
            "symbol_id",
            "symbol_type",
            "start_line",
            "scope_type",
            "scope_id",
            "parent_scope_id",
        ]
    ]
    .sort_values(["file_id", "local_name", "start_line"])
    .to_string(index=False)
)

'                      file_id local_name  export_start_line            symbol_id symbol_type  start_line scope_type               scope_id        parent_scope_id\n      resource/loader.sys.mjs       data                373 sym_dc22f24ff5de4a2e    variable        81.0      block scope_50b001a411b7bc42 scope_2764b0f7bccbd955\n      resource/loader.sys.mjs       data                373 sym_5e0d17e41d8f3ddb    variable       364.0      block scope_5f40024f26517cae scope_6a92a6d1b017e989\ntest/resource/pako_inflate.js   GZheader                461 sym_e7d49d5947899024    function       425.0   function scope_356dd0f61d8a59e1 scope_1357a46cc6da1ef6\ntest/resource/pako_inflate.js   GZheader                461 sym_3d5579ae5197b27b    variable      2681.0      block scope_f726acd48853de3b scope_02238dcb1c24f24f\ntest/resource/pako_inflate.js    ZStream               2669 sym_cfd0ee028be241f9    function      2644.0   function scope_eefa7d22b603477f scope_21ce4e02147cd86f\ntest/resource/pako_in

In [105]:
# Build a scope lookup
scope_lookup = scopes_df.set_index("scope_id").to_dict("index")


def get_scope_ancestors(scope_id):
    ancestors = []
    current = scope_id
    visited = set()

    while current and current not in visited:
        visited.add(current)

        scope = scope_lookup.get(current)

        if scope is None:
            break

        ancestors.append({
            "scope_id": current,
            "scope_type": scope.get("scope_type"),
            "parent_scope_id": scope.get("parent_scope_id"),
        })

        current = scope.get("parent_scope_id")

    return ancestors


# Show candidate scope ancestry
scope_chain_records = []

for row in unresolved_detail_df.itertuples(index=False):

    for depth, ancestor in enumerate(
        get_scope_ancestors(row.scope_id)
    ):
        scope_chain_records.append({
            "file_id": row.file_id,
            "local_name": row.local_name,
            "export_start_line": row.export_start_line,
            "symbol_id": row.symbol_id,
            "candidate_line": row.start_line,
            "depth": depth,
            **ancestor,
        })

scope_chains_df = pd.DataFrame(scope_chain_records)

display(
    scope_chains_df[
        scope_chains_df["local_name"].isin(
            ["data", "GZheader", "ZStream", "inflate"]
        )
    ].to_string(index=False)
)

'                      file_id local_name  export_start_line            symbol_id  candidate_line  depth               scope_id scope_type        parent_scope_id\n      resource/loader.sys.mjs       data                373 sym_dc22f24ff5de4a2e            81.0      0 scope_50b001a411b7bc42      block scope_2764b0f7bccbd955\n      resource/loader.sys.mjs       data                373 sym_dc22f24ff5de4a2e            81.0      1 scope_2764b0f7bccbd955   function scope_9c12fcc6d14e6f2a\n      resource/loader.sys.mjs       data                373 sym_dc22f24ff5de4a2e            81.0      2 scope_9c12fcc6d14e6f2a     module                    NaN\n      resource/loader.sys.mjs       data                373 sym_5e0d17e41d8f3ddb           364.0      0 scope_5f40024f26517cae      block scope_6a92a6d1b017e989\n      resource/loader.sys.mjs       data                373 sym_5e0d17e41d8f3ddb           364.0      1 scope_6a92a6d1b017e989      block scope_f801b22c115eec25\n      resource/loader.sys.m

In [106]:
# Inspect scope boundaries around unresolved export assignments

scope_boundary_records = []

for row in unresolved_detail_df[
    ["file_id", "local_name", "export_start_line"]
].drop_duplicates().itertuples(index=False):

    file_id = row.file_id
    export_line = int(row.export_start_line)

    file_scopes = scopes_df[
        scopes_df["file_id"] == file_id
    ].copy()

    # Find scopes whose line range contains the export assignment
    containing_scopes = file_scopes[
        (file_scopes["start_line"] <= export_line)
        & (file_scopes["end_line"] >= export_line)
    ].copy()

    containing_scopes["span"] = (
        containing_scopes["end_line"]
        - containing_scopes["start_line"]
    )

    containing_scopes = containing_scopes.sort_values("span")

    for scope in containing_scopes.itertuples(index=False):
        scope_boundary_records.append({
            "file_id": file_id,
            "local_name": row.local_name,
            "export_line": export_line,
            "scope_id": scope.scope_id,
            "scope_type": scope.scope_type,
            "start_line": scope.start_line,
            "end_line": scope.end_line,
            "span": scope.span,
            "parent_scope_id": scope.parent_scope_id,
        })

export_scope_candidates_df = pd.DataFrame(scope_boundary_records)

display(export_scope_candidates_df.to_string(index=False))

'                      file_id local_name  export_line               scope_id scope_type  start_line  end_line  span        parent_scope_id\n      resource/loader.sys.mjs       data          373 scope_fd7b055540997d91      block         370       374     4 scope_5f40024f26517cae\n      resource/loader.sys.mjs       data          373 scope_5f40024f26517cae      block         363       382    19 scope_6a92a6d1b017e989\n      resource/loader.sys.mjs       data          373 scope_f801b22c115eec25   function         341       402    61 scope_0240735b3c194a38\n      resource/loader.sys.mjs       data          373 scope_6a92a6d1b017e989      block         341       402    61 scope_f801b22c115eec25\n      resource/loader.sys.mjs       data          373 scope_1b85054733fe8458   function         321       469   148 scope_9c12fcc6d14e6f2a\n      resource/loader.sys.mjs       data          373 scope_0240735b3c194a38      block         321       469   148 scope_1b85054733fe8458\n      resource/load

In [107]:
def is_scope_ancestor(ancestor_scope_id, child_scope_id):
    current = child_scope_id
    visited = set()

    while current and current not in visited:
        if current == ancestor_scope_id:
            return True

        visited.add(current)
        scope = scope_lookup.get(current)

        if scope is None:
            break

        current = scope.get("parent_scope_id")

    return False


def resolve_by_scope(group):

    candidates = group.drop_duplicates("symbol_id").copy()

    export_scope_id = group["export_scope_id"].iloc[0]
    export_line = group["export_start_line"].iloc[0]

    visible = []

    for candidate in candidates.itertuples(index=False):

        candidate_scope = candidate.scope_id
        declaration_type = candidate.declaration_type

        # The candidate's scope must enclose the export scope.
        scope_match = is_scope_ancestor(
            candidate_scope,
            export_scope_id
        )

        # Function declarations are hoisted; lexical variables are not.
        if declaration_type in (
            "function_declaration",
        ):
            position_match = True
        else:
            position_match = candidate.start_line <= export_line

        if scope_match and position_match:
            visible.append(candidate)

    if len(visible) == 1:
        return pd.Series({
            "resolved_symbol_id": visible[0].symbol_id,
            "resolution_status": "resolved",
            "resolution_basis": "scope_visibility",
        })

    return pd.Series({
        "resolved_symbol_id": None,
        "resolution_status": (
            "ambiguous" if len(visible) > 1 else "unresolved"
        ),
        "resolution_basis": (
            "multiple_visible_candidates"
            if len(visible) > 1
            else "no_visible_candidate"
        ),
    })

In [109]:
export_scope_lookup = (
    export_scope_candidates_df
    .sort_values("span")
    .drop_duplicates(["file_id", "local_name", "export_line"])
)

export_scope_lookup = export_scope_lookup.rename(columns={
    "scope_id": "export_scope_id"
})

scope_aware_candidates_df = unresolved_detail_df.merge(
    export_scope_lookup[
        ["file_id", "local_name", "export_line", "export_scope_id"]
    ],
    left_on=["file_id", "local_name", "export_start_line"],
    right_on=["file_id", "local_name", "export_line"],
    how="left",
)

scope_aware_candidates_df["start_line"] = (
    pd.to_numeric(scope_aware_candidates_df["start_line"])
)

In [111]:
scope_resolution_df = (
    scope_aware_candidates_df
    .groupby("export_binding_id", group_keys=False)
    .apply(resolve_by_scope)
    .reset_index()
)

display(
    scope_resolution_df["resolution_status"]
    .value_counts()
    .to_frame("count")
)

display(
    scope_resolution_df[
        scope_resolution_df["resolution_status"] != "resolved"
    ]
)

,count
resolution_status,
unresolved,7
resolved,1


,export_binding_id,resolved_symbol_id,resolution_status,resolution_basis
0,exb_006683bac1b974f6,NaN,unresolved,no_visible_candidate
1,exb_2d345ae5a3eb1827,NaN,unresolved,no_visible_candidate
2,exb_369c98c82f55805e,NaN,unresolved,no_visible_candidate
3,exb_667da70c81909d62,NaN,unresolved,no_visible_candidate
4,exb_7107b1c0090b3a8c,NaN,unresolved,no_visible_candidate
5,exb_abbd8c54655bb4ba,NaN,unresolved,no_visible_candidate
6,exb_bdfd970dd9cbe6c3,NaN,unresolved,no_visible_candidate


,count
resolution_status,
unresolved,7
resolved,1


,export_binding_id,resolved_symbol_id,resolution_status,resolution_basis
0,exb_006683bac1b974f6,NaN,unresolved,no_visible_candidate
1,exb_2d345ae5a3eb1827,NaN,unresolved,no_visible_candidate
2,exb_369c98c82f55805e,NaN,unresolved,no_visible_candidate
3,exb_667da70c81909d62,NaN,unresolved,no_visible_candidate
4,exb_7107b1c0090b3a8c,NaN,unresolved,no_visible_candidate
5,exb_abbd8c54655bb4ba,NaN,unresolved,no_visible_candidate
6,exb_bdfd970dd9cbe6c3,NaN,unresolved,no_visible_candidate


In [112]:
display(
    scope_aware_candidates_df[
        scope_aware_candidates_df["export_binding_id"].isin(
            scope_resolution_df.loc[
                scope_resolution_df["resolution_status"] == "resolved",
                "export_binding_id"
            ]
        )
    ][[
        "file_id",
        "local_name",
        "export_start_line",
        "symbol_type",
        "declaration_type",
        "start_line",
        "scope_id",
        "export_scope_id",
    ]]
)

,file_id,local_name,export_start_line,symbol_type,declaration_type,start_line,scope_id,export_scope_id
0,resource/loader.sys.mjs,data,373,variable,lexical_declaration,81.0,scope_50b001a411b7bc42,scope_fd7b055540997d91
1,resource/loader.sys.mjs,data,373,variable,lexical_declaration,364.0,scope_5f40024f26517cae,scope_fd7b055540997d91


In [113]:
function_scope_check_df = (
    symbols_df[
        symbols_df["declaration_type"] == "function_declaration"
    ]
    .merge(
        scopes_df[
            ["scope_id", "scope_type", "parent_scope_id",
             "start_line", "end_line"]
        ],
        on="scope_id",
        how="left",
        suffixes=("_symbol", "_scope"),
    )
)

display(
    function_scope_check_df[
        [
            "file_id",
            "name",
            "start_line_symbol",
            "scope_id",
            "scope_type",
            "parent_scope_id",
            "start_line_scope",
            "end_line_scope",
        ]
    ].head(15)
)

,file_id,name,start_line_symbol,scope_id,scope_type,parent_scope_id,start_line_scope,end_line_scope
0,chrome/content/scaffold/content.js,onLoad,11,scope_1143e81631668e56,function,scope_b4b9e4b0b1d63790,11,13
1,chrome/content/scaffold/scaffold.js,_getMetadataObject,825,scope_6e484fa15c4d9a1c,function,scope_a3a4caaab41ee805,825,882
2,chrome/content/scaffold/scaffold.js,_getCode,884,scope_f7e53e8722613a93,function,scope_a3a4caaab41ee805,884,891
3,chrome/content/scaffold/scaffold.js,_run,1185,scope_f1560cc6414a0830,function,scope_a3a4caaab41ee805,1185,1274
4,chrome/content/scaffold/scaffold.js,_testTargetRegex,1304,scope_f2051f1f1241f55c,function,scope_a3a4caaab41ee805,1304,1315
5,chrome/content/scaffold/scaffold.js,_selectItems,1320,scope_42fef2f619071a6e,function,scope_a3a4caaab41ee805,1320,1325
6,chrome/content/scaffold/scaffold.js,_error,1330,scope_e7f46ab01d7d8514,function,scope_a3a4caaab41ee805,1330,1332
7,chrome/content/scaffold/scaffold.js,_debug,1337,scope_a2328e82edb27b65,function,scope_a3a4caaab41ee805,1337,1339
8,chrome/content/scaffold/scaffold.js,_myItemDone,1344,scope_4a0dbcf64953b483,function,scope_a3a4caaab41ee805,1344,1347
9,chrome/content/scaffold/scaffold.js,_myExportDone,1352,scope_3ab210e54e78acf9,function,scope_a3a4caaab41ee805,1352,1360


In [114]:
symbol_scope_lookup = (
    scopes_df.set_index("scope_id").to_dict("index")
)

def get_declaration_scope(row):

    scope_id = row["scope_id"]

    if row["declaration_type"] == "function_declaration":
        scope = symbol_scope_lookup.get(scope_id, {})
        return scope.get("parent_scope_id")

    return scope_id


symbols_for_resolution_df = symbols_df.copy()

symbols_for_resolution_df["declaration_scope_id"] = (
    symbols_for_resolution_df.apply(
        get_declaration_scope,
        axis=1
    )
)

In [115]:
display(
    symbols_for_resolution_df[
        symbols_for_resolution_df["declaration_type"]
        == "function_declaration"
    ][
        [
            "file_id",
            "name",
            "start_line",
            "scope_id",
            "declaration_scope_id",
        ]
    ].head(15)
)

,file_id,name,start_line,scope_id,declaration_scope_id
11,chrome/content/scaffold/content.js,onLoad,11,scope_1143e81631668e56,scope_b4b9e4b0b1d63790
197,chrome/content/scaffold/scaffold.js,_getMetadataObject,825,scope_6e484fa15c4d9a1c,scope_a3a4caaab41ee805
200,chrome/content/scaffold/scaffold.js,_getCode,884,scope_f7e53e8722613a93,scope_a3a4caaab41ee805
261,chrome/content/scaffold/scaffold.js,_run,1185,scope_f1560cc6414a0830,scope_a3a4caaab41ee805
270,chrome/content/scaffold/scaffold.js,_testTargetRegex,1304,scope_f2051f1f1241f55c,scope_a3a4caaab41ee805
273,chrome/content/scaffold/scaffold.js,_selectItems,1320,scope_42fef2f619071a6e,scope_a3a4caaab41ee805
275,chrome/content/scaffold/scaffold.js,_error,1330,scope_e7f46ab01d7d8514,scope_a3a4caaab41ee805
276,chrome/content/scaffold/scaffold.js,_debug,1337,scope_a2328e82edb27b65,scope_a3a4caaab41ee805
277,chrome/content/scaffold/scaffold.js,_myItemDone,1344,scope_4a0dbcf64953b483,scope_a3a4caaab41ee805
279,chrome/content/scaffold/scaffold.js,_myExportDone,1352,scope_3ab210e54e78acf9,scope_a3a4caaab41ee805


In [117]:
scope_aware_candidates_df = (
    named_bindings_df
    .merge(
        symbols_for_resolution_df,
        left_on=["file_id", "local_name"],
        right_on=["file_id", "name"],
        how="left",
        suffixes=("_export", "_symbol"),
    )
    .merge(
        export_scope_lookup[
            ["file_id", "local_name", "export_line", "export_scope_id"]
        ],
        left_on=["file_id", "local_name", "start_line"],
        right_on=["file_id", "local_name", "export_line"],
        how="left",
    )
)

###### 8. Identifier references and assignments

In [118]:
def extract_references(file_id: str, tree, source_code: bytes) -> pd.DataFrame:
    records = []
    for node in walk_tree(tree.root_node):
        if node.type not in {"identifier", "shorthand_property_identifier"}:
            continue

        parent = node.parent
        # Exclude obvious declaration/property-name positions. Resolution can refine
        # this classification later; the raw source evidence remains available.
        if parent and parent.type in {
            "variable_declarator", "function_declaration",
            "class_declaration", "method_definition",
            "import_specifier", "formal_parameters"
        }:
            continue

        role = "reference"
        if parent and parent.type == "call_expression":
            role = "call_callee"
        elif parent and parent.type == "assignment_expression":
            role = "assignment_target" if parent.named_children and parent.named_children[
                0] == node else "assignment_expression"
        elif parent and parent.type == "member_expression" and parent.named_children and parent.named_children[
            -1] == node:
            role = "member_property"

        loc = node_location(node)
        records.append({
            "reference_id": stable_id(file_id, node.start_byte, prefix="ref_"),
            "file_id": file_id,
            "name": node_text(node, source_code),
            "reference_role": role,
            "scope_id": None,
            "resolved_symbol_id": None,
            **loc,
        })
    return pd.DataFrame(records)


def extract_assignments(file_id: str, tree, source_code: bytes) -> pd.DataFrame:
    records = []
    for node in walk_tree(tree.root_node):
        if node.type not in {"assignment_expression", "augmented_assignment_expression", "variable_declarator"}:
            continue
        children = node.named_children
        target = children[0] if children else None
        value = children[-1] if len(children) > 1 else None
        loc = node_location(node)
        records.append({
            "assignment_id": stable_id(file_id, node.start_byte, prefix="asg_"),
            "file_id": file_id,
            "assignment_kind": node.type,
            "target_expression": node_text(target, source_code) if target else None,
            "value_expression": node_text(value, source_code) if value and value != target else None,
            **loc,
        })
    return pd.DataFrame(records)


references_df = pd.concat(
    [extract_references(fid, tree, source_by_file[fid]) for fid, tree in ast_trees.items()],
    ignore_index=True
) if ast_trees else pd.DataFrame()

assignments_df = pd.concat(
    [extract_assignments(fid, tree, source_by_file[fid]) for fid, tree in ast_trees.items()],
    ignore_index=True
) if ast_trees else pd.DataFrame()

display(references_df.head(30))
display(assignments_df.head(30))


,reference_id,file_id,name,reference_role,scope_id,resolved_symbol_id,start_line,start_column,end_line,end_column,start_byte,end_byte
0,ref_6a0ce5fabc0d788e,app/assets/commandLineHandler.js,ChromeUtils,reference,None,None,1,43,1,54,42,53
1,ref_f274b1f8b0893350,app/assets/commandLineHandler.js,cmdLine,reference,None,None,5,5,5,12,262,269
2,ref_b1d050d716f004e8,app/assets/commandLineHandler.js,Ci,reference,None,None,5,22,5,24,279,281
3,ref_9d9636f34aed2aa1,app/assets/commandLineHandler.js,cmdLine,reference,None,None,6,2,6,9,322,329
4,ref_80f5d9a64f1c4b0b,app/assets/commandLineHandler.js,cmdLine,reference,None,None,10,5,10,12,392,399
5,ref_878bff57d642f3f6,app/assets/commandLineHandler.js,CommandLineOptions,reference,None,None,11,2,11,20,437,455
6,ref_96a87ec634a4b578,app/assets/commandLineHandler.js,cmdLine,reference,None,None,14,10,14,17,524,531
7,ref_348ffe0ecd591af6,app/assets/commandLineHandler.js,CommandLineOptions,reference,None,None,15,2,15,20,573,591
8,ref_a41c796f6fe1bbbb,app/assets/commandLineHandler.js,CommandLineOptions,reference,None,None,20,5,20,23,898,916
9,ref_22adff4180b67460,app/assets/commandLineHandler.js,Services,reference,None,None,21,2,21,10,935,943


,assignment_id,file_id,assignment_kind,target_expression,value_expression,start_line,start_column,end_line,end_column,start_byte,end_byte
0,asg_688a35739fce99cc,app/assets/commandLineHandler.js,variable_declarator,"{ CommandLineOptions, TestOptions }","ChromeUtils.importESModule(""chrome://zotero/co...",1,5,1,127,4,126
1,asg_9d9636f34aed2aa1,app/assets/commandLineHandler.js,assignment_expression,cmdLine.preventDefault,true,6,2,6,31,322,351
2,asg_878bff57d642f3f6,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDebugLog,2,11,2,11,38,437,473
3,asg_348ffe0ecd591af6,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDebugLog,1,15,2,15,38,573,609
4,asg_559ddbdd7eb6dbd6,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.forceDataDir,"cmdLine.handleFlagWithParam(""datadir"", false)",24,1,24,80,1016,1095
5,asg_71ad3a1356db8a14,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.file,"cmdLine.handleFlagWithParam(""file"", false)",26,1,26,69,1158,1226
6,asg_46de9770a57e8f6a,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.url,"cmdLine.handleFlagWithParam(""url"", false)",27,1,27,67,1228,1294
7,asg_18b1bc890b11b3ec,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.url,cmdLine.resolveURI(CommandLineOptions.url),29,2,29,69,1327,1394
8,asg_b47b92a4792dec83,app/assets/commandLineHandler.js,variable_declarator,processTestOptions,false,32,5,32,31,1403,1429
9,asg_e4e507887ce443fe,app/assets/commandLineHandler.js,assignment_expression,CommandLineOptions.test,true,34,2,34,32,1479,1509


In [120]:
file_id = "app/assets/commandLineHandler.js"

display(
    references_df[
        references_df["file_id"] == file_id
    ][
        [
            "name",
            "reference_role",
            "start_line",
            "start_column",
        ]
    ].head(40)
)

display(
    assignments_df[
        assignments_df["file_id"] == file_id
    ][
        [
            "assignment_kind",
            "target_expression",
            "value_expression",
            "start_line",
        ]
    ].head(15)
)

,name,reference_role,start_line,start_column
0,ChromeUtils,reference,1,43
1,cmdLine,reference,5,5
2,Ci,reference,5,22
3,cmdLine,reference,6,2
4,cmdLine,reference,10,5
5,CommandLineOptions,reference,11,2
6,cmdLine,reference,14,10
7,CommandLineOptions,reference,15,2
8,CommandLineOptions,reference,20,5
9,Services,reference,21,2


,assignment_kind,target_expression,value_expression,start_line
0,variable_declarator,"{ CommandLineOptions, TestOptions }","ChromeUtils.importESModule(""chrome://zotero/co...",1
1,assignment_expression,cmdLine.preventDefault,true,6
2,assignment_expression,CommandLineOptions.forceDebugLog,2,11
3,assignment_expression,CommandLineOptions.forceDebugLog,1,15
4,assignment_expression,CommandLineOptions.forceDataDir,"cmdLine.handleFlagWithParam(""datadir"", false)",24
5,assignment_expression,CommandLineOptions.file,"cmdLine.handleFlagWithParam(""file"", false)",26
6,assignment_expression,CommandLineOptions.url,"cmdLine.handleFlagWithParam(""url"", false)",27
7,assignment_expression,CommandLineOptions.url,cmdLine.resolveURI(CommandLineOptions.url),29
8,variable_declarator,processTestOptions,false,32
9,assignment_expression,CommandLineOptions.test,true,34
